In [1]:
# -*- coding: utf-8 -*-
"""
Created on Mon Mar  2 12:08:39 2026

LR SESUAI LR FINDER

@author: indri
"""

# =========================================================
# CLEAN RESEARCH PIPELINE — DISTILBERT BASELINE
# =========================================================

import os, json, random, time, csv
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch import amp
from torch.optim.lr_scheduler import ReduceLROnPlateau
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
from tqdm import tqdm


# ================= CONFIG =================
CONFIG = {
    "model_name":"distilbert-base-uncased",
    "max_len":128,
    "batch_size":32,
    "epochs":50,
    "lr":2.88e-06,
    "patience":4,
    "seed":42,
    "num_labels":2
}
OUTPUT_DIR = rf"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/DISTILBERT_BASELINE/1.ARW.{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(OUTPUT_DIR, exist_ok=True)
json.dump(CONFIG, open(f"{OUTPUT_DIR}/config.json","w"), indent=4)

DATA_PATH = r"/content/drive/MyDrive/Colab Notebooks/PREPROCESS/1.RAW.csv"

# ================= REPRO =================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ================= AUGMENTASI =================
def text_augment(text):
    tokens = text.split()
    if len(tokens) < 5: return text # Jangan ganggu kalimat pendek

    r = random.random()
    # Random Swap
    if r < 0.15:
        idx1, idx2 = random.sample(range(len(tokens)), 2)
        tokens[idx1], tokens[idx2] = tokens[idx2], tokens[idx1]
    # Random Delete
    elif r < 0.30:
        tokens = [t for t in tokens if random.random() > 0.1]

    return " ".join(tokens)
# ================= TOKENIZER =================
tokenizer = DistilBertTokenizer.from_pretrained(CONFIG["model_name"])

# ================= DATASET =================
class TextDataset(Dataset):
    def __init__(self, texts, labels, augment=False):
        self.texts = texts
        self.labels = labels
        self.augment = augment # Tambahkan flag ini

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        # Jalankan augmentasi hanya jika flag Aktif
        if self.augment:
            text = text_augment(text)

        enc = tokenizer(
            text, # Teks yang sudah (mungkin) diaugmentasi
            truncation=True,
            padding="max_length",
            max_length=CONFIG["max_len"],
            return_tensors="pt"
        )

        return (
            enc["input_ids"].squeeze(0),
            enc["attention_mask"].squeeze(0),
            torch.tensor(self.labels[idx], dtype=torch.long)
        )

# ================= LOAD DATA =================
df = pd.read_csv(DATA_PATH).dropna()
"""
df["sentiment"] = (
    df["sentiment"]
    .astype(str).str.strip().str.lower()
    .replace({"0":"negative","1":"positive"})
    .map({"negative":0,"positive":1})
)

texts = df["review"].astype(str).tolist()
labels = df["sentiment"].tolist()

train_idx = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\train.npy")
val_idx   = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\val.npy")
test_idx  = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\test.npy")

def subset(arr, ids): return [arr[i] for i in ids]
"""
all_reviews = df['review'].astype(str).values
all_labels = df["sentiment"].replace({"0":0, "1":1}).values # Pastikan integer

train_idx = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/train.npy")
val_idx   = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/val.npy")
test_idx  = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/test.npy")

# 2. Distribusikan berdasarkan index yang sudah di-load (Anti-Leakage)
train_texts = [all_reviews[i] for i in train_idx]
train_labels = [all_labels[i] for i in train_idx]

val_texts = [all_reviews[i] for i in val_idx]
val_labels = [all_labels[i] for i in val_idx]

test_texts = [all_reviews[i] for i in test_idx]
test_labels = [all_labels[i] for i in test_idx]

# 3. Verifikasi Kebocoran (Tambahkan ini untuk bukti di Tesis)
assert len(set(train_idx) & set(test_idx)) == 0, "LEAKAGE DETECTED: Train & Test overlap!"
assert len(set(train_idx) & set(val_idx)) == 0, "LEAKAGE DETECTED: Train & Val overlap!"

#==============================================
#KALAU MAU PAKE SAMPLE
#==============================================

DEBUG = False
DEBUG_RATIO = 0.05

if DEBUG:
    train_idx = train_idx[:int(len(train_idx)*DEBUG_RATIO)]
    val_idx   = val_idx[:int(len(val_idx)*DEBUG_RATIO)]
    test_idx  = test_idx[:int(len(test_idx)*DEBUG_RATIO)]

#===============================================
# HANYA train_ds yang boleh augmentasi
"""
train_ds = TextDataset(subset(texts,train_idx), subset(labels,train_idx), augment=True)

# val dan test HARUS augment=False (default)
val_ds   = TextDataset(subset(texts,val_idx),   subset(labels,val_idx), augment=False)
test_ds  = TextDataset(subset(texts,test_idx),  subset(labels,test_idx), augment=False)

train_loader = DataLoader(train_ds,batch_size=CONFIG["batch_size"],shuffle=True)
val_loader   = DataLoader(val_ds,batch_size=CONFIG["batch_size"])
test_loader  = DataLoader(test_ds,batch_size=CONFIG["batch_size"])
"""
# 3. Distribusikan data (Gunakan List Comprehension seperti di BiGRU)
train_texts = [all_reviews[i] for i in train_idx]
train_labels = [all_labels[i] for i in train_idx]

val_texts = [all_reviews[i] for i in val_idx]
val_labels = [all_labels[i] for i in val_idx]

test_texts = [all_reviews[i] for i in test_idx]
test_labels = [all_labels[i] for i in test_idx]

# 4. Inisialisasi Dataset yang BENAR (Hapus vocab, samakan nama class)
train_dataset = TextDataset(train_texts, train_labels, augment=True)
val_dataset   = TextDataset(val_texts, val_labels, augment=False)
test_dataset  = TextDataset(test_texts, test_labels, augment=False)

g = torch.Generator()
g.manual_seed(CONFIG["seed"])

def seed_worker(worker_id):
    worker_seed = (CONFIG["seed"]) + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

train_loader = DataLoader(train_dataset,
                          batch_size=(CONFIG["batch_size"]),
                          shuffle=True,
                          worker_init_fn=seed_worker,
                          generator=g)
val_loader = DataLoader(val_dataset, batch_size=(CONFIG["batch_size"]))
test_loader = DataLoader(test_dataset, batch_size=(CONFIG["batch_size"]))

# ================= MODEL =================
class DistilBERTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained(CONFIG["model_name"])
        self.fc = nn.Linear(768, CONFIG["num_labels"])

    def forward(self, input_ids, attn):
        out = self.bert(input_ids=input_ids, attention_mask=attn)
        cls = out.last_hidden_state[:,0]      # CLS pooling (standard baseline)
        return self.fc(cls)

model = DistilBERTClassifier().to(device)

# ================= OPTIM =================
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=1e-4)
current_lr = optimizer.param_groups[0]["lr"]
print(f"LR = {current_lr:.2e}")
scheduler = ReduceLROnPlateau(optimizer, mode="max", patience=2, factor=0.5)
criterion = nn.CrossEntropyLoss()
scaler = amp.GradScaler(enabled=torch.cuda.is_available())

# ================= TRAIN =================
def train_epoch():
    model.train()
    total_loss, preds, gold = 0, [], []

    for ids,attn,labels in tqdm(train_loader, desc="Training", leave=False):
        ids,attn,labels = ids.to(device), attn.to(device), labels.to(device)

        optimizer.zero_grad()

        with amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
            logits = model(ids,attn)
            loss = criterion(logits,labels)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        preds.extend(torch.argmax(logits,1).cpu().numpy())
        gold.extend(labels.cpu().numpy())

    return total_loss/len(train_loader), preds, gold

# ================= EVAL =================
def evaluate(loader):
    model.eval()
    total_loss, preds, gold, probs = 0, [], [], []

    with torch.no_grad():
        for ids,attn,labels in tqdm(loader, desc="Evaluating", leave=False):
            ids,attn,labels = ids.to(device), attn.to(device), labels.to(device)

            logits = model(ids,attn)
            loss = criterion(logits,labels)

            total_loss += loss.item()

            p = torch.softmax(logits,1)
            preds.extend(torch.argmax(p,1).cpu().numpy())
            gold.extend(labels.cpu().numpy())
            probs.extend(p[:,1].cpu().numpy())

    return total_loss/len(loader), preds, gold, probs


CHECKPOINT_PATH = f"{OUTPUT_DIR}/checkpoint.pt"

start_epoch = 0
best_acc = 0

if os.path.exists(CHECKPOINT_PATH):
    print("Loading checkpoint...")
    ckpt = torch.load(CHECKPOINT_PATH)

    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])

    start_epoch = ckpt["epoch"] + 1
    best_acc = ckpt["best_acc"]

    random.setstate(ckpt["python_rng"])
    np.random.set_state(ckpt["numpy_rng"])
    torch.set_rng_state(ckpt["torch_rng"])
# ================= TRAIN LOOP =================
best_acc = 0
patience = 0
history = []
total_start = time.time()
train_start = time.time()
for epoch in range(start_epoch, CONFIG["epochs"]):

    tr_loss,tr_p,tr_g = train_epoch()
    val_loss,val_p,val_g,_ = evaluate(val_loader)

    #print(f"Epoch {epoch+1} | train {tr_loss:.4f} | val {val_loss:.4f}")
    tr_acc = accuracy_score(tr_g, tr_p)
    val_acc = accuracy_score(val_g, val_p)
    history.append([epoch+1,tr_loss,val_loss,tr_acc,val_acc])

    scheduler.step(val_acc)

    print(f"Epoch {epoch+1} | "
          f"train_loss {tr_loss:.4f} | val_loss {val_loss:.4f} | "
          f"train_acc {tr_acc:.4f} | val_acc {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        patience = 0
        torch.save(model.state_dict(), f"{OUTPUT_DIR}/best.pt")
    else:
        patience += 1
        if patience >= CONFIG["patience"]:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break # INI YANG KURANG!
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "best_acc": best_acc,
        "python_rng": random.getstate(),
        "numpy_rng": np.random.get_state(),
        "torch_rng": torch.get_rng_state()
    }, CHECKPOINT_PATH)

train_time = time.time() - train_start
# ================= TEST =================
test_start = time.time()
model.load_state_dict(torch.load(f"{OUTPUT_DIR}/best.pt"))

test_loss,test_p,test_g,test_prob = evaluate(test_loader)

acc  = accuracy_score(test_g,test_p)
prec = precision_score(test_g,test_p)
rec  = recall_score(test_g,test_p)
f1   = f1_score(test_g,test_p)

print("\nTEST RESULT")
print(acc,prec,rec,f1)
test_time = time.time() - test_start
total_time = time.time() - total_start
# ================= TEST =================
# ... (kode evaluasi Anda)

# 1. HITUNG WAKTU SEBELUM SIMPAN
inference_time_total = test_time
num_samples = len(test_texts)
avg_inference_per_sample = (inference_time_total / num_samples) * 1000

# 2. SIMPAN METRICS (SATU KALI SAJA - JANGAN DOUBLE)
metrics_data = {
    "accuracy": acc,
    "precision": prec,
    "recall": rec,
    "f1": f1,
    "loss": test_loss,
    "time": {
        "train_time_sec": train_time,
        "test_time_sec": test_time,
        "avg_inference_ms": avg_inference_per_sample,
        "total_time_sec": total_time
    }
}

with open(f"{OUTPUT_DIR}/metrics.json", "w") as f:
    json.dump(metrics_data, f, indent=4)

# HAPUS BLOK "# ================= SAVE METRICS =================" BERIKUTNYA


# ================= CURVES =================
epochs=[x[0] for x in history]

plt.plot(epochs,[x[1] for x in history],label="train")
plt.plot(epochs,[x[2] for x in history],label="val")
plt.legend(); plt.grid()
plt.savefig(f"{OUTPUT_DIR}/loss_curve.png")
plt.close()

plt.plot(epochs,[x[3] for x in history],label="train acc")
plt.plot(epochs,[x[4] for x in history],label="val acc")
plt.legend(); plt.grid()
plt.savefig(f"{OUTPUT_DIR}/accuracy_curve.png")
plt.close()


# ================= ROC =================
fpr,tpr,_ = roc_curve(test_g,test_prob)
roc_auc = auc(fpr,tpr)

plt.plot(fpr,tpr,label=f"AUC={roc_auc:.4f}")
plt.plot([0,1],[0,1],'--')
plt.legend(); plt.grid()
plt.savefig(f"{OUTPUT_DIR}/roc.png")
plt.close()


# ================= CM =================
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(test_g, test_p)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative","Positive"]
)

plt.figure(figsize=(6,6))
disp.plot(values_format="d")
plt.title("Confusion Matrix")
plt.grid(False)
plt.savefig(f"{OUTPUT_DIR}/confusion_matrix.png")
plt.close()

# ================= SAVE TRAINING LOGS (CSV) =================
log_df = pd.DataFrame(history, columns=["epoch", "train_loss", "val_loss", "train_acc", "val_acc"])
log_df.to_csv(f"{OUTPUT_DIR}/training_log.csv", index=False)

# ================= SAVE DETAILED CLASSIFICATION REPORT =================
from sklearn.metrics import classification_report
report = classification_report(test_g, test_p, target_names=["Negative", "Positive"], output_dict=True)
pd.DataFrame(report).transpose().to_csv(f"{OUTPUT_DIR}/classification_report.csv")

# ================= SAVE PREDICTIONS (Lebih Lengkap untuk Analisis) =================
pred_df = pd.DataFrame({
    "text": test_texts,        # Tambahkan ini agar bisa dibaca manusia
    "actual": test_g,
    "predicted": test_p,
    "probability": test_prob
})
pred_df.to_csv(f"{OUTPUT_DIR}/test_predictions.csv", index=False)

<>:120: SyntaxWarning: invalid escape sequence '\T'
<>:120: SyntaxWarning: invalid escape sequence '\T'
/tmp/ipykernel_796/1841205107.py:120: SyntaxWarning: invalid escape sequence '\T'
  train_idx = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\train.npy")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

...

Epoch 27 | train_loss 0.1171 | val_loss 0.3827 | train_acc 0.9576 | val_acc 0.8864


Epoch 28 | train_loss 0.1059 | val_loss 0.4248 | train_acc 0.9627 | val_acc 0.8828


Epoch 29 | train_loss 0.1018 | val_loss 0.4149 | train_acc 0.9642 | val_acc 0.8834
Early stopping triggered at epoch 29



TEST RESULT
0.8832 0.8835068054443554 0.8828 0.8831532613045218


<Figure size 600x600 with 0 Axes>

# **2**

In [2]:
# -*- coding: utf-8 -*-
"""
Created on Mon Mar  2 12:08:39 2026

LR SESUAI LR FINDER

@author: indri
"""

# =========================================================
# CLEAN RESEARCH PIPELINE — DISTILBERT BASELINE
# =========================================================

import os, json, random, time, csv
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch import amp
from torch.optim.lr_scheduler import ReduceLROnPlateau
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
from tqdm import tqdm


# ================= CONFIG =================
CONFIG = {
    "model_name":"distilbert-base-uncased",
    "max_len":128,
    "batch_size":32,
    "epochs":50,
    "lr":2.88e-06,
    "patience":4,
    "seed":42,
    "num_labels":2
}
OUTPUT_DIR = rf"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/DISTILBERT_BASELINE/2.L.{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(OUTPUT_DIR, exist_ok=True)
json.dump(CONFIG, open(f"{OUTPUT_DIR}/config.json","w"), indent=4)

DATA_PATH = r"/content/drive/MyDrive/Colab Notebooks/PREPROCESS/2.L.csv"

# ================= REPRO =================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ================= AUGMENTASI =================
def text_augment(text):
    tokens = text.split()
    if len(tokens) < 5: return text # Jangan ganggu kalimat pendek

    r = random.random()
    # Random Swap
    if r < 0.15:
        idx1, idx2 = random.sample(range(len(tokens)), 2)
        tokens[idx1], tokens[idx2] = tokens[idx2], tokens[idx1]
    # Random Delete
    elif r < 0.30:
        tokens = [t for t in tokens if random.random() > 0.1]

    return " ".join(tokens)
# ================= TOKENIZER =================
tokenizer = DistilBertTokenizer.from_pretrained(CONFIG["model_name"])

# ================= DATASET =================
class TextDataset(Dataset):
    def __init__(self, texts, labels, augment=False):
        self.texts = texts
        self.labels = labels
        self.augment = augment # Tambahkan flag ini

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        # Jalankan augmentasi hanya jika flag Aktif
        if self.augment:
            text = text_augment(text)

        enc = tokenizer(
            text, # Teks yang sudah (mungkin) diaugmentasi
            truncation=True,
            padding="max_length",
            max_length=CONFIG["max_len"],
            return_tensors="pt"
        )

        return (
            enc["input_ids"].squeeze(0),
            enc["attention_mask"].squeeze(0),
            torch.tensor(self.labels[idx], dtype=torch.long)
        )

# ================= LOAD DATA =================
df = pd.read_csv(DATA_PATH).dropna()
"""
df["sentiment"] = (
    df["sentiment"]
    .astype(str).str.strip().str.lower()
    .replace({"0":"negative","1":"positive"})
    .map({"negative":0,"positive":1})
)

texts = df["review"].astype(str).tolist()
labels = df["sentiment"].tolist()

train_idx = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\train.npy")
val_idx   = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\val.npy")
test_idx  = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\test.npy")

def subset(arr, ids): return [arr[i] for i in ids]
"""
all_reviews = df['review'].astype(str).values
all_labels = df["sentiment"].replace({"0":0, "1":1}).values # Pastikan integer

train_idx = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/train.npy")
val_idx   = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/val.npy")
test_idx  = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/test.npy")

# 2. Distribusikan berdasarkan index yang sudah di-load (Anti-Leakage)
train_texts = [all_reviews[i] for i in train_idx]
train_labels = [all_labels[i] for i in train_idx]

val_texts = [all_reviews[i] for i in val_idx]
val_labels = [all_labels[i] for i in val_idx]

test_texts = [all_reviews[i] for i in test_idx]
test_labels = [all_labels[i] for i in test_idx]

# 3. Verifikasi Kebocoran (Tambahkan ini untuk bukti di Tesis)
assert len(set(train_idx) & set(test_idx)) == 0, "LEAKAGE DETECTED: Train & Test overlap!"
assert len(set(train_idx) & set(val_idx)) == 0, "LEAKAGE DETECTED: Train & Val overlap!"

#==============================================
#KALAU MAU PAKE SAMPLE
#==============================================

DEBUG = False
DEBUG_RATIO = 0.05

if DEBUG:
    train_idx = train_idx[:int(len(train_idx)*DEBUG_RATIO)]
    val_idx   = val_idx[:int(len(val_idx)*DEBUG_RATIO)]
    test_idx  = test_idx[:int(len(test_idx)*DEBUG_RATIO)]

#===============================================
# HANYA train_ds yang boleh augmentasi
"""
train_ds = TextDataset(subset(texts,train_idx), subset(labels,train_idx), augment=True)

# val dan test HARUS augment=False (default)
val_ds   = TextDataset(subset(texts,val_idx),   subset(labels,val_idx), augment=False)
test_ds  = TextDataset(subset(texts,test_idx),  subset(labels,test_idx), augment=False)

train_loader = DataLoader(train_ds,batch_size=CONFIG["batch_size"],shuffle=True)
val_loader   = DataLoader(val_ds,batch_size=CONFIG["batch_size"])
test_loader  = DataLoader(test_ds,batch_size=CONFIG["batch_size"])
"""
# 3. Distribusikan data (Gunakan List Comprehension seperti di BiGRU)
train_texts = [all_reviews[i] for i in train_idx]
train_labels = [all_labels[i] for i in train_idx]

val_texts = [all_reviews[i] for i in val_idx]
val_labels = [all_labels[i] for i in val_idx]

test_texts = [all_reviews[i] for i in test_idx]
test_labels = [all_labels[i] for i in test_idx]

# 4. Inisialisasi Dataset yang BENAR (Hapus vocab, samakan nama class)
train_dataset = TextDataset(train_texts, train_labels, augment=True)
val_dataset   = TextDataset(val_texts, val_labels, augment=False)
test_dataset  = TextDataset(test_texts, test_labels, augment=False)

g = torch.Generator()
g.manual_seed(CONFIG["seed"])

def seed_worker(worker_id):
    worker_seed = (CONFIG["seed"]) + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

train_loader = DataLoader(train_dataset,
                          batch_size=(CONFIG["batch_size"]),
                          shuffle=True,
                          worker_init_fn=seed_worker,
                          generator=g)
val_loader = DataLoader(val_dataset, batch_size=(CONFIG["batch_size"]))
test_loader = DataLoader(test_dataset, batch_size=(CONFIG["batch_size"]))

# ================= MODEL =================
class DistilBERTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained(CONFIG["model_name"])
        self.fc = nn.Linear(768, CONFIG["num_labels"])

    def forward(self, input_ids, attn):
        out = self.bert(input_ids=input_ids, attention_mask=attn)
        cls = out.last_hidden_state[:,0]      # CLS pooling (standard baseline)
        return self.fc(cls)

model = DistilBERTClassifier().to(device)

# ================= OPTIM =================
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=1e-4)
current_lr = optimizer.param_groups[0]["lr"]
print(f"LR = {current_lr:.2e}")
scheduler = ReduceLROnPlateau(optimizer, mode="max", patience=2, factor=0.5)
criterion = nn.CrossEntropyLoss()
scaler = amp.GradScaler(enabled=torch.cuda.is_available())

# ================= TRAIN =================
def train_epoch():
    model.train()
    total_loss, preds, gold = 0, [], []

    for ids,attn,labels in tqdm(train_loader, desc="Training", leave=False):
        ids,attn,labels = ids.to(device), attn.to(device), labels.to(device)

        optimizer.zero_grad()

        with amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
            logits = model(ids,attn)
            loss = criterion(logits,labels)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        preds.extend(torch.argmax(logits,1).cpu().numpy())
        gold.extend(labels.cpu().numpy())

    return total_loss/len(train_loader), preds, gold

# ================= EVAL =================
def evaluate(loader):
    model.eval()
    total_loss, preds, gold, probs = 0, [], [], []

    with torch.no_grad():
        for ids,attn,labels in tqdm(loader, desc="Evaluating", leave=False):
            ids,attn,labels = ids.to(device), attn.to(device), labels.to(device)

            logits = model(ids,attn)
            loss = criterion(logits,labels)

            total_loss += loss.item()

            p = torch.softmax(logits,1)
            preds.extend(torch.argmax(p,1).cpu().numpy())
            gold.extend(labels.cpu().numpy())
            probs.extend(p[:,1].cpu().numpy())

    return total_loss/len(loader), preds, gold, probs


CHECKPOINT_PATH = f"{OUTPUT_DIR}/checkpoint.pt"

start_epoch = 0
best_acc = 0

if os.path.exists(CHECKPOINT_PATH):
    print("Loading checkpoint...")
    ckpt = torch.load(CHECKPOINT_PATH)

    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])

    start_epoch = ckpt["epoch"] + 1
    best_acc = ckpt["best_acc"]

    random.setstate(ckpt["python_rng"])
    np.random.set_state(ckpt["numpy_rng"])
    torch.set_rng_state(ckpt["torch_rng"])
# ================= TRAIN LOOP =================
best_acc = 0
patience = 0
history = []
total_start = time.time()
train_start = time.time()
for epoch in range(start_epoch, CONFIG["epochs"]):

    tr_loss,tr_p,tr_g = train_epoch()
    val_loss,val_p,val_g,_ = evaluate(val_loader)

    #print(f"Epoch {epoch+1} | train {tr_loss:.4f} | val {val_loss:.4f}")
    tr_acc = accuracy_score(tr_g, tr_p)
    val_acc = accuracy_score(val_g, val_p)
    history.append([epoch+1,tr_loss,val_loss,tr_acc,val_acc])

    scheduler.step(val_acc)

    print(f"Epoch {epoch+1} | "
          f"train_loss {tr_loss:.4f} | val_loss {val_loss:.4f} | "
          f"train_acc {tr_acc:.4f} | val_acc {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        patience = 0
        torch.save(model.state_dict(), f"{OUTPUT_DIR}/best.pt")
    else:
        patience += 1
        if patience >= CONFIG["patience"]:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break # INI YANG KURANG!
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "best_acc": best_acc,
        "python_rng": random.getstate(),
        "numpy_rng": np.random.get_state(),
        "torch_rng": torch.get_rng_state()
    }, CHECKPOINT_PATH)

train_time = time.time() - train_start
# ================= TEST =================
test_start = time.time()
model.load_state_dict(torch.load(f"{OUTPUT_DIR}/best.pt"))

test_loss,test_p,test_g,test_prob = evaluate(test_loader)

acc  = accuracy_score(test_g,test_p)
prec = precision_score(test_g,test_p)
rec  = recall_score(test_g,test_p)
f1   = f1_score(test_g,test_p)

print("\nTEST RESULT")
print(acc,prec,rec,f1)
test_time = time.time() - test_start
total_time = time.time() - total_start
# ================= TEST =================
# ... (kode evaluasi Anda)

# 1. HITUNG WAKTU SEBELUM SIMPAN
inference_time_total = test_time
num_samples = len(test_texts)
avg_inference_per_sample = (inference_time_total / num_samples) * 1000

# 2. SIMPAN METRICS (SATU KALI SAJA - JANGAN DOUBLE)
metrics_data = {
    "accuracy": acc,
    "precision": prec,
    "recall": rec,
    "f1": f1,
    "loss": test_loss,
    "time": {
        "train_time_sec": train_time,
        "test_time_sec": test_time,
        "avg_inference_ms": avg_inference_per_sample,
        "total_time_sec": total_time
    }
}

with open(f"{OUTPUT_DIR}/metrics.json", "w") as f:
    json.dump(metrics_data, f, indent=4)

# HAPUS BLOK "# ================= SAVE METRICS =================" BERIKUTNYA


# ================= CURVES =================
epochs=[x[0] for x in history]

plt.plot(epochs,[x[1] for x in history],label="train")
plt.plot(epochs,[x[2] for x in history],label="val")
plt.legend(); plt.grid()
plt.savefig(f"{OUTPUT_DIR}/loss_curve.png")
plt.close()

plt.plot(epochs,[x[3] for x in history],label="train acc")
plt.plot(epochs,[x[4] for x in history],label="val acc")
plt.legend(); plt.grid()
plt.savefig(f"{OUTPUT_DIR}/accuracy_curve.png")
plt.close()


# ================= ROC =================
fpr,tpr,_ = roc_curve(test_g,test_prob)
roc_auc = auc(fpr,tpr)

plt.plot(fpr,tpr,label=f"AUC={roc_auc:.4f}")
plt.plot([0,1],[0,1],'--')
plt.legend(); plt.grid()
plt.savefig(f"{OUTPUT_DIR}/roc.png")
plt.close()


# ================= CM =================
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(test_g, test_p)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative","Positive"]
)

plt.figure(figsize=(6,6))
disp.plot(values_format="d")
plt.title("Confusion Matrix")
plt.grid(False)
plt.savefig(f"{OUTPUT_DIR}/confusion_matrix.png")
plt.close()

# ================= SAVE TRAINING LOGS (CSV) =================
log_df = pd.DataFrame(history, columns=["epoch", "train_loss", "val_loss", "train_acc", "val_acc"])
log_df.to_csv(f"{OUTPUT_DIR}/training_log.csv", index=False)

# ================= SAVE DETAILED CLASSIFICATION REPORT =================
from sklearn.metrics import classification_report
report = classification_report(test_g, test_p, target_names=["Negative", "Positive"], output_dict=True)
pd.DataFrame(report).transpose().to_csv(f"{OUTPUT_DIR}/classification_report.csv")

# ================= SAVE PREDICTIONS (Lebih Lengkap untuk Analisis) =================
pred_df = pd.DataFrame({
    "text": test_texts,        # Tambahkan ini agar bisa dibaca manusia
    "actual": test_g,
    "predicted": test_p,
    "probability": test_prob
})
pred_df.to_csv(f"{OUTPUT_DIR}/test_predictions.csv", index=False)

<>:120: SyntaxWarning: invalid escape sequence '\T'
<>:120: SyntaxWarning: invalid escape sequence '\T'
/tmp/ipykernel_796/1416476797.py:120: SyntaxWarning: invalid escape sequence '\T'
  train_idx = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\train.npy")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LR = 2.88e-06


Epoch 1 | train_loss 0.5003 | val_loss 0.3808 | train_acc 0.7557 | val_acc 0.8270


...

Epoch 21 | train_loss 0.1713 | val_loss 0.3298 | train_acc 0.9342 | val_acc 0.8802


Epoch 22 | train_loss 0.1624 | val_loss 0.3423 | train_acc 0.9379 | val_acc 0.8758


Epoch 23 | train_loss 0.1537 | val_loss 0.3495 | train_acc 0.9416 | val_acc 0.8738
Early stopping triggered at epoch 23



TEST RESULT
0.877 0.8842233999184672 0.8676 0.8758328285887341


<Figure size 600x600 with 0 Axes>

# **3**

In [3]:
# -*- coding: utf-8 -*-
"""
Created on Mon Mar  2 12:08:39 2026

LR SESUAI LR FINDER

@author: indri
"""

# =========================================================
# CLEAN RESEARCH PIPELINE — DISTILBERT BASELINE
# =========================================================

import os, json, random, time, csv
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch import amp
from torch.optim.lr_scheduler import ReduceLROnPlateau
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
from tqdm import tqdm


# ================= CONFIG =================
CONFIG = {
    "model_name":"distilbert-base-uncased",
    "max_len":128,
    "batch_size":32,
    "epochs":50,
    "lr":2.88e-06,
    "patience":4,
    "seed":42,
    "num_labels":2
}
OUTPUT_DIR = rf"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/DISTILBERT_BASELINE/3.R.{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(OUTPUT_DIR, exist_ok=True)
json.dump(CONFIG, open(f"{OUTPUT_DIR}/config.json","w"), indent=4)

DATA_PATH = r"/content/drive/MyDrive/Colab Notebooks/PREPROCESS/3.R.csv"

# ================= REPRO =================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ================= AUGMENTASI =================
def text_augment(text):
    tokens = text.split()
    if len(tokens) < 5: return text # Jangan ganggu kalimat pendek

    r = random.random()
    # Random Swap
    if r < 0.15:
        idx1, idx2 = random.sample(range(len(tokens)), 2)
        tokens[idx1], tokens[idx2] = tokens[idx2], tokens[idx1]
    # Random Delete
    elif r < 0.30:
        tokens = [t for t in tokens if random.random() > 0.1]

    return " ".join(tokens)
# ================= TOKENIZER =================
tokenizer = DistilBertTokenizer.from_pretrained(CONFIG["model_name"])

# ================= DATASET =================
class TextDataset(Dataset):
    def __init__(self, texts, labels, augment=False):
        self.texts = texts
        self.labels = labels
        self.augment = augment # Tambahkan flag ini

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        # Jalankan augmentasi hanya jika flag Aktif
        if self.augment:
            text = text_augment(text)

        enc = tokenizer(
            text, # Teks yang sudah (mungkin) diaugmentasi
            truncation=True,
            padding="max_length",
            max_length=CONFIG["max_len"],
            return_tensors="pt"
        )

        return (
            enc["input_ids"].squeeze(0),
            enc["attention_mask"].squeeze(0),
            torch.tensor(self.labels[idx], dtype=torch.long)
        )

# ================= LOAD DATA =================
df = pd.read_csv(DATA_PATH).dropna()
"""
df["sentiment"] = (
    df["sentiment"]
    .astype(str).str.strip().str.lower()
    .replace({"0":"negative","1":"positive"})
    .map({"negative":0,"positive":1})
)

texts = df["review"].astype(str).tolist()
labels = df["sentiment"].tolist()

train_idx = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\train.npy")
val_idx   = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\val.npy")
test_idx  = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\test.npy")

def subset(arr, ids): return [arr[i] for i in ids]
"""
all_reviews = df['review'].astype(str).values
all_labels = df["sentiment"].replace({"0":0, "1":1}).values # Pastikan integer

train_idx = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/train.npy")
val_idx   = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/val.npy")
test_idx  = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/test.npy")

# 2. Distribusikan berdasarkan index yang sudah di-load (Anti-Leakage)
train_texts = [all_reviews[i] for i in train_idx]
train_labels = [all_labels[i] for i in train_idx]

val_texts = [all_reviews[i] for i in val_idx]
val_labels = [all_labels[i] for i in val_idx]

test_texts = [all_reviews[i] for i in test_idx]
test_labels = [all_labels[i] for i in test_idx]

# 3. Verifikasi Kebocoran (Tambahkan ini untuk bukti di Tesis)
assert len(set(train_idx) & set(test_idx)) == 0, "LEAKAGE DETECTED: Train & Test overlap!"
assert len(set(train_idx) & set(val_idx)) == 0, "LEAKAGE DETECTED: Train & Val overlap!"

#==============================================
#KALAU MAU PAKE SAMPLE
#==============================================

DEBUG = False
DEBUG_RATIO = 0.05

if DEBUG:
    train_idx = train_idx[:int(len(train_idx)*DEBUG_RATIO)]
    val_idx   = val_idx[:int(len(val_idx)*DEBUG_RATIO)]
    test_idx  = test_idx[:int(len(test_idx)*DEBUG_RATIO)]

#===============================================
# HANYA train_ds yang boleh augmentasi
"""
train_ds = TextDataset(subset(texts,train_idx), subset(labels,train_idx), augment=True)

# val dan test HARUS augment=False (default)
val_ds   = TextDataset(subset(texts,val_idx),   subset(labels,val_idx), augment=False)
test_ds  = TextDataset(subset(texts,test_idx),  subset(labels,test_idx), augment=False)

train_loader = DataLoader(train_ds,batch_size=CONFIG["batch_size"],shuffle=True)
val_loader   = DataLoader(val_ds,batch_size=CONFIG["batch_size"])
test_loader  = DataLoader(test_ds,batch_size=CONFIG["batch_size"])
"""
# 3. Distribusikan data (Gunakan List Comprehension seperti di BiGRU)
train_texts = [all_reviews[i] for i in train_idx]
train_labels = [all_labels[i] for i in train_idx]

val_texts = [all_reviews[i] for i in val_idx]
val_labels = [all_labels[i] for i in val_idx]

test_texts = [all_reviews[i] for i in test_idx]
test_labels = [all_labels[i] for i in test_idx]

# 4. Inisialisasi Dataset yang BENAR (Hapus vocab, samakan nama class)
train_dataset = TextDataset(train_texts, train_labels, augment=True)
val_dataset   = TextDataset(val_texts, val_labels, augment=False)
test_dataset  = TextDataset(test_texts, test_labels, augment=False)

g = torch.Generator()
g.manual_seed(CONFIG["seed"])

def seed_worker(worker_id):
    worker_seed = (CONFIG["seed"]) + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

train_loader = DataLoader(train_dataset,
                          batch_size=(CONFIG["batch_size"]),
                          shuffle=True,
                          worker_init_fn=seed_worker,
                          generator=g)
val_loader = DataLoader(val_dataset, batch_size=(CONFIG["batch_size"]))
test_loader = DataLoader(test_dataset, batch_size=(CONFIG["batch_size"]))

# ================= MODEL =================
class DistilBERTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained(CONFIG["model_name"])
        self.fc = nn.Linear(768, CONFIG["num_labels"])

    def forward(self, input_ids, attn):
        out = self.bert(input_ids=input_ids, attention_mask=attn)
        cls = out.last_hidden_state[:,0]      # CLS pooling (standard baseline)
        return self.fc(cls)

model = DistilBERTClassifier().to(device)

# ================= OPTIM =================
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=1e-4)
current_lr = optimizer.param_groups[0]["lr"]
print(f"LR = {current_lr:.2e}")
scheduler = ReduceLROnPlateau(optimizer, mode="max", patience=2, factor=0.5)
criterion = nn.CrossEntropyLoss()
scaler = amp.GradScaler(enabled=torch.cuda.is_available())

# ================= TRAIN =================
def train_epoch():
    model.train()
    total_loss, preds, gold = 0, [], []

    for ids,attn,labels in tqdm(train_loader, desc="Training", leave=False):
        ids,attn,labels = ids.to(device), attn.to(device), labels.to(device)

        optimizer.zero_grad()

        with amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
            logits = model(ids,attn)
            loss = criterion(logits,labels)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        preds.extend(torch.argmax(logits,1).cpu().numpy())
        gold.extend(labels.cpu().numpy())

    return total_loss/len(train_loader), preds, gold

# ================= EVAL =================
def evaluate(loader):
    model.eval()
    total_loss, preds, gold, probs = 0, [], [], []

    with torch.no_grad():
        for ids,attn,labels in tqdm(loader, desc="Evaluating", leave=False):
            ids,attn,labels = ids.to(device), attn.to(device), labels.to(device)

            logits = model(ids,attn)
            loss = criterion(logits,labels)

            total_loss += loss.item()

            p = torch.softmax(logits,1)
            preds.extend(torch.argmax(p,1).cpu().numpy())
            gold.extend(labels.cpu().numpy())
            probs.extend(p[:,1].cpu().numpy())

    return total_loss/len(loader), preds, gold, probs


CHECKPOINT_PATH = f"{OUTPUT_DIR}/checkpoint.pt"

start_epoch = 0
best_acc = 0

if os.path.exists(CHECKPOINT_PATH):
    print("Loading checkpoint...")
    ckpt = torch.load(CHECKPOINT_PATH)

    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])

    start_epoch = ckpt["epoch"] + 1
    best_acc = ckpt["best_acc"]

    random.setstate(ckpt["python_rng"])
    np.random.set_state(ckpt["numpy_rng"])
    torch.set_rng_state(ckpt["torch_rng"])
# ================= TRAIN LOOP =================
best_acc = 0
patience = 0
history = []
total_start = time.time()
train_start = time.time()
for epoch in range(start_epoch, CONFIG["epochs"]):

    tr_loss,tr_p,tr_g = train_epoch()
    val_loss,val_p,val_g,_ = evaluate(val_loader)

    #print(f"Epoch {epoch+1} | train {tr_loss:.4f} | val {val_loss:.4f}")
    tr_acc = accuracy_score(tr_g, tr_p)
    val_acc = accuracy_score(val_g, val_p)
    history.append([epoch+1,tr_loss,val_loss,tr_acc,val_acc])

    scheduler.step(val_acc)

    print(f"Epoch {epoch+1} | "
          f"train_loss {tr_loss:.4f} | val_loss {val_loss:.4f} | "
          f"train_acc {tr_acc:.4f} | val_acc {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        patience = 0
        torch.save(model.state_dict(), f"{OUTPUT_DIR}/best.pt")
    else:
        patience += 1
        if patience >= CONFIG["patience"]:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break # INI YANG KURANG!
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "best_acc": best_acc,
        "python_rng": random.getstate(),
        "numpy_rng": np.random.get_state(),
        "torch_rng": torch.get_rng_state()
    }, CHECKPOINT_PATH)

train_time = time.time() - train_start
# ================= TEST =================
test_start = time.time()
model.load_state_dict(torch.load(f"{OUTPUT_DIR}/best.pt"))

test_loss,test_p,test_g,test_prob = evaluate(test_loader)

acc  = accuracy_score(test_g,test_p)
prec = precision_score(test_g,test_p)
rec  = recall_score(test_g,test_p)
f1   = f1_score(test_g,test_p)

print("\nTEST RESULT")
print(acc,prec,rec,f1)
test_time = time.time() - test_start
total_time = time.time() - total_start
# ================= TEST =================
# ... (kode evaluasi Anda)

# 1. HITUNG WAKTU SEBELUM SIMPAN
inference_time_total = test_time
num_samples = len(test_texts)
avg_inference_per_sample = (inference_time_total / num_samples) * 1000

# 2. SIMPAN METRICS (SATU KALI SAJA - JANGAN DOUBLE)
metrics_data = {
    "accuracy": acc,
    "precision": prec,
    "recall": rec,
    "f1": f1,
    "loss": test_loss,
    "time": {
        "train_time_sec": train_time,
        "test_time_sec": test_time,
        "avg_inference_ms": avg_inference_per_sample,
        "total_time_sec": total_time
    }
}

with open(f"{OUTPUT_DIR}/metrics.json", "w") as f:
    json.dump(metrics_data, f, indent=4)

# HAPUS BLOK "# ================= SAVE METRICS =================" BERIKUTNYA


# ================= CURVES =================
epochs=[x[0] for x in history]

plt.plot(epochs,[x[1] for x in history],label="train")
plt.plot(epochs,[x[2] for x in history],label="val")
plt.legend(); plt.grid()
plt.savefig(f"{OUTPUT_DIR}/loss_curve.png")
plt.close()

plt.plot(epochs,[x[3] for x in history],label="train acc")
plt.plot(epochs,[x[4] for x in history],label="val acc")
plt.legend(); plt.grid()
plt.savefig(f"{OUTPUT_DIR}/accuracy_curve.png")
plt.close()


# ================= ROC =================
fpr,tpr,_ = roc_curve(test_g,test_prob)
roc_auc = auc(fpr,tpr)

plt.plot(fpr,tpr,label=f"AUC={roc_auc:.4f}")
plt.plot([0,1],[0,1],'--')
plt.legend(); plt.grid()
plt.savefig(f"{OUTPUT_DIR}/roc.png")
plt.close()


# ================= CM =================
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(test_g, test_p)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative","Positive"]
)

plt.figure(figsize=(6,6))
disp.plot(values_format="d")
plt.title("Confusion Matrix")
plt.grid(False)
plt.savefig(f"{OUTPUT_DIR}/confusion_matrix.png")
plt.close()

# ================= SAVE TRAINING LOGS (CSV) =================
log_df = pd.DataFrame(history, columns=["epoch", "train_loss", "val_loss", "train_acc", "val_acc"])
log_df.to_csv(f"{OUTPUT_DIR}/training_log.csv", index=False)

# ================= SAVE DETAILED CLASSIFICATION REPORT =================
from sklearn.metrics import classification_report
report = classification_report(test_g, test_p, target_names=["Negative", "Positive"], output_dict=True)
pd.DataFrame(report).transpose().to_csv(f"{OUTPUT_DIR}/classification_report.csv")

# ================= SAVE PREDICTIONS (Lebih Lengkap untuk Analisis) =================
pred_df = pd.DataFrame({
    "text": test_texts,        # Tambahkan ini agar bisa dibaca manusia
    "actual": test_g,
    "predicted": test_p,
    "probability": test_prob
})
pred_df.to_csv(f"{OUTPUT_DIR}/test_predictions.csv", index=False)

<>:120: SyntaxWarning: invalid escape sequence '\T'
<>:120: SyntaxWarning: invalid escape sequence '\T'
/tmp/ipykernel_796/2378567112.py:120: SyntaxWarning: invalid escape sequence '\T'
  train_idx = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\train.npy")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LR = 2.88e-06


Epoch 1 | train_loss 0.4857 | val_loss 0.3625 | train_acc 0.7683 | val_acc 0.8382


...

Epoch 14 | train_loss 0.2080 | val_loss 0.3012 | train_acc 0.9182 | val_acc 0.8770


Epoch 15 | train_loss 0.2003 | val_loss 0.3027 | train_acc 0.9211 | val_acc 0.8788


Epoch 16 | train_loss 0.1921 | val_loss 0.3046 | train_acc 0.9257 | val_acc 0.8808
Early stopping triggered at epoch 16



TEST RESULT
0.881 0.869615832363213 0.8964 0.8828048059877881


<Figure size 600x600 with 0 Axes>

# **4**

In [4]:
# -*- coding: utf-8 -*-
"""
Created on Mon Mar  2 12:08:39 2026

LR SESUAI LR FINDER

@author: indri
"""

# =========================================================
# CLEAN RESEARCH PIPELINE — DISTILBERT BASELINE
# =========================================================

import os, json, random, time, csv
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch import amp
from torch.optim.lr_scheduler import ReduceLROnPlateau
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
from tqdm import tqdm


# ================= CONFIG =================
CONFIG = {
    "model_name":"distilbert-base-uncased",
    "max_len":128,
    "batch_size":32,
    "epochs":50,
    "lr":2.88e-06,
    "patience":4,
    "seed":42,
    "num_labels":2
}
OUTPUT_DIR = rf"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/DISTILBERT_BASELINE/4.S.{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(OUTPUT_DIR, exist_ok=True)
json.dump(CONFIG, open(f"{OUTPUT_DIR}/config.json","w"), indent=4)

DATA_PATH = r"/content/drive/MyDrive/Colab Notebooks/PREPROCESS/4.S.csv"

# ================= REPRO =================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ================= AUGMENTASI =================
def text_augment(text):
    tokens = text.split()
    if len(tokens) < 5: return text # Jangan ganggu kalimat pendek

    r = random.random()
    # Random Swap
    if r < 0.15:
        idx1, idx2 = random.sample(range(len(tokens)), 2)
        tokens[idx1], tokens[idx2] = tokens[idx2], tokens[idx1]
    # Random Delete
    elif r < 0.30:
        tokens = [t for t in tokens if random.random() > 0.1]

    return " ".join(tokens)
# ================= TOKENIZER =================
tokenizer = DistilBertTokenizer.from_pretrained(CONFIG["model_name"])

# ================= DATASET =================
class TextDataset(Dataset):
    def __init__(self, texts, labels, augment=False):
        self.texts = texts
        self.labels = labels
        self.augment = augment # Tambahkan flag ini

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        # Jalankan augmentasi hanya jika flag Aktif
        if self.augment:
            text = text_augment(text)

        enc = tokenizer(
            text, # Teks yang sudah (mungkin) diaugmentasi
            truncation=True,
            padding="max_length",
            max_length=CONFIG["max_len"],
            return_tensors="pt"
        )

        return (
            enc["input_ids"].squeeze(0),
            enc["attention_mask"].squeeze(0),
            torch.tensor(self.labels[idx], dtype=torch.long)
        )

# ================= LOAD DATA =================
df = pd.read_csv(DATA_PATH).dropna()
"""
df["sentiment"] = (
    df["sentiment"]
    .astype(str).str.strip().str.lower()
    .replace({"0":"negative","1":"positive"})
    .map({"negative":0,"positive":1})
)

texts = df["review"].astype(str).tolist()
labels = df["sentiment"].tolist()

train_idx = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\train.npy")
val_idx   = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\val.npy")
test_idx  = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\test.npy")

def subset(arr, ids): return [arr[i] for i in ids]
"""
all_reviews = df['review'].astype(str).values
all_labels = df["sentiment"].replace({"0":0, "1":1}).values # Pastikan integer

train_idx = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/train.npy")
val_idx   = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/val.npy")
test_idx  = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/test.npy")

# 2. Distribusikan berdasarkan index yang sudah di-load (Anti-Leakage)
train_texts = [all_reviews[i] for i in train_idx]
train_labels = [all_labels[i] for i in train_idx]

val_texts = [all_reviews[i] for i in val_idx]
val_labels = [all_labels[i] for i in val_idx]

test_texts = [all_reviews[i] for i in test_idx]
test_labels = [all_labels[i] for i in test_idx]

# 3. Verifikasi Kebocoran (Tambahkan ini untuk bukti di Tesis)
assert len(set(train_idx) & set(test_idx)) == 0, "LEAKAGE DETECTED: Train & Test overlap!"
assert len(set(train_idx) & set(val_idx)) == 0, "LEAKAGE DETECTED: Train & Val overlap!"

#==============================================
#KALAU MAU PAKE SAMPLE
#==============================================

DEBUG = False
DEBUG_RATIO = 0.05

if DEBUG:
    train_idx = train_idx[:int(len(train_idx)*DEBUG_RATIO)]
    val_idx   = val_idx[:int(len(val_idx)*DEBUG_RATIO)]
    test_idx  = test_idx[:int(len(test_idx)*DEBUG_RATIO)]

#===============================================
# HANYA train_ds yang boleh augmentasi
"""
train_ds = TextDataset(subset(texts,train_idx), subset(labels,train_idx), augment=True)

# val dan test HARUS augment=False (default)
val_ds   = TextDataset(subset(texts,val_idx),   subset(labels,val_idx), augment=False)
test_ds  = TextDataset(subset(texts,test_idx),  subset(labels,test_idx), augment=False)

train_loader = DataLoader(train_ds,batch_size=CONFIG["batch_size"],shuffle=True)
val_loader   = DataLoader(val_ds,batch_size=CONFIG["batch_size"])
test_loader  = DataLoader(test_ds,batch_size=CONFIG["batch_size"])
"""
# 3. Distribusikan data (Gunakan List Comprehension seperti di BiGRU)
train_texts = [all_reviews[i] for i in train_idx]
train_labels = [all_labels[i] for i in train_idx]

val_texts = [all_reviews[i] for i in val_idx]
val_labels = [all_labels[i] for i in val_idx]

test_texts = [all_reviews[i] for i in test_idx]
test_labels = [all_labels[i] for i in test_idx]

# 4. Inisialisasi Dataset yang BENAR (Hapus vocab, samakan nama class)
train_dataset = TextDataset(train_texts, train_labels, augment=True)
val_dataset   = TextDataset(val_texts, val_labels, augment=False)
test_dataset  = TextDataset(test_texts, test_labels, augment=False)

g = torch.Generator()
g.manual_seed(CONFIG["seed"])

def seed_worker(worker_id):
    worker_seed = (CONFIG["seed"]) + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

train_loader = DataLoader(train_dataset,
                          batch_size=(CONFIG["batch_size"]),
                          shuffle=True,
                          worker_init_fn=seed_worker,
                          generator=g)
val_loader = DataLoader(val_dataset, batch_size=(CONFIG["batch_size"]))
test_loader = DataLoader(test_dataset, batch_size=(CONFIG["batch_size"]))

# ================= MODEL =================
class DistilBERTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained(CONFIG["model_name"])
        self.fc = nn.Linear(768, CONFIG["num_labels"])

    def forward(self, input_ids, attn):
        out = self.bert(input_ids=input_ids, attention_mask=attn)
        cls = out.last_hidden_state[:,0]      # CLS pooling (standard baseline)
        return self.fc(cls)

model = DistilBERTClassifier().to(device)

# ================= OPTIM =================
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=1e-4)
current_lr = optimizer.param_groups[0]["lr"]
print(f"LR = {current_lr:.2e}")
scheduler = ReduceLROnPlateau(optimizer, mode="max", patience=2, factor=0.5)
criterion = nn.CrossEntropyLoss()
scaler = amp.GradScaler(enabled=torch.cuda.is_available())

# ================= TRAIN =================
def train_epoch():
    model.train()
    total_loss, preds, gold = 0, [], []

    for ids,attn,labels in tqdm(train_loader, desc="Training", leave=False):
        ids,attn,labels = ids.to(device), attn.to(device), labels.to(device)

        optimizer.zero_grad()

        with amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
            logits = model(ids,attn)
            loss = criterion(logits,labels)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        preds.extend(torch.argmax(logits,1).cpu().numpy())
        gold.extend(labels.cpu().numpy())

    return total_loss/len(train_loader), preds, gold

# ================= EVAL =================
def evaluate(loader):
    model.eval()
    total_loss, preds, gold, probs = 0, [], [], []

    with torch.no_grad():
        for ids,attn,labels in tqdm(loader, desc="Evaluating", leave=False):
            ids,attn,labels = ids.to(device), attn.to(device), labels.to(device)

            logits = model(ids,attn)
            loss = criterion(logits,labels)

            total_loss += loss.item()

            p = torch.softmax(logits,1)
            preds.extend(torch.argmax(p,1).cpu().numpy())
            gold.extend(labels.cpu().numpy())
            probs.extend(p[:,1].cpu().numpy())

    return total_loss/len(loader), preds, gold, probs


CHECKPOINT_PATH = f"{OUTPUT_DIR}/checkpoint.pt"

start_epoch = 0
best_acc = 0

if os.path.exists(CHECKPOINT_PATH):
    print("Loading checkpoint...")
    ckpt = torch.load(CHECKPOINT_PATH)

    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])

    start_epoch = ckpt["epoch"] + 1
    best_acc = ckpt["best_acc"]

    random.setstate(ckpt["python_rng"])
    np.random.set_state(ckpt["numpy_rng"])
    torch.set_rng_state(ckpt["torch_rng"])
# ================= TRAIN LOOP =================
best_acc = 0
patience = 0
history = []
total_start = time.time()
train_start = time.time()
for epoch in range(start_epoch, CONFIG["epochs"]):

    tr_loss,tr_p,tr_g = train_epoch()
    val_loss,val_p,val_g,_ = evaluate(val_loader)

    #print(f"Epoch {epoch+1} | train {tr_loss:.4f} | val {val_loss:.4f}")
    tr_acc = accuracy_score(tr_g, tr_p)
    val_acc = accuracy_score(val_g, val_p)
    history.append([epoch+1,tr_loss,val_loss,tr_acc,val_acc])

    scheduler.step(val_acc)

    print(f"Epoch {epoch+1} | "
          f"train_loss {tr_loss:.4f} | val_loss {val_loss:.4f} | "
          f"train_acc {tr_acc:.4f} | val_acc {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        patience = 0
        torch.save(model.state_dict(), f"{OUTPUT_DIR}/best.pt")
    else:
        patience += 1
        if patience >= CONFIG["patience"]:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break # INI YANG KURANG!
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "best_acc": best_acc,
        "python_rng": random.getstate(),
        "numpy_rng": np.random.get_state(),
        "torch_rng": torch.get_rng_state()
    }, CHECKPOINT_PATH)

train_time = time.time() - train_start
# ================= TEST =================
test_start = time.time()
model.load_state_dict(torch.load(f"{OUTPUT_DIR}/best.pt"))

test_loss,test_p,test_g,test_prob = evaluate(test_loader)

acc  = accuracy_score(test_g,test_p)
prec = precision_score(test_g,test_p)
rec  = recall_score(test_g,test_p)
f1   = f1_score(test_g,test_p)

print("\nTEST RESULT")
print(acc,prec,rec,f1)
test_time = time.time() - test_start
total_time = time.time() - total_start
# ================= TEST =================
# ... (kode evaluasi Anda)

# 1. HITUNG WAKTU SEBELUM SIMPAN
inference_time_total = test_time
num_samples = len(test_texts)
avg_inference_per_sample = (inference_time_total / num_samples) * 1000

# 2. SIMPAN METRICS (SATU KALI SAJA - JANGAN DOUBLE)
metrics_data = {
    "accuracy": acc,
    "precision": prec,
    "recall": rec,
    "f1": f1,
    "loss": test_loss,
    "time": {
        "train_time_sec": train_time,
        "test_time_sec": test_time,
        "avg_inference_ms": avg_inference_per_sample,
        "total_time_sec": total_time
    }
}

with open(f"{OUTPUT_DIR}/metrics.json", "w") as f:
    json.dump(metrics_data, f, indent=4)

# HAPUS BLOK "# ================= SAVE METRICS =================" BERIKUTNYA


# ================= CURVES =================
epochs=[x[0] for x in history]

plt.plot(epochs,[x[1] for x in history],label="train")
plt.plot(epochs,[x[2] for x in history],label="val")
plt.legend(); plt.grid()
plt.savefig(f"{OUTPUT_DIR}/loss_curve.png")
plt.close()

plt.plot(epochs,[x[3] for x in history],label="train acc")
plt.plot(epochs,[x[4] for x in history],label="val acc")
plt.legend(); plt.grid()
plt.savefig(f"{OUTPUT_DIR}/accuracy_curve.png")
plt.close()


# ================= ROC =================
fpr,tpr,_ = roc_curve(test_g,test_prob)
roc_auc = auc(fpr,tpr)

plt.plot(fpr,tpr,label=f"AUC={roc_auc:.4f}")
plt.plot([0,1],[0,1],'--')
plt.legend(); plt.grid()
plt.savefig(f"{OUTPUT_DIR}/roc.png")
plt.close()


# ================= CM =================
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(test_g, test_p)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative","Positive"]
)

plt.figure(figsize=(6,6))
disp.plot(values_format="d")
plt.title("Confusion Matrix")
plt.grid(False)
plt.savefig(f"{OUTPUT_DIR}/confusion_matrix.png")
plt.close()

# ================= SAVE TRAINING LOGS (CSV) =================
log_df = pd.DataFrame(history, columns=["epoch", "train_loss", "val_loss", "train_acc", "val_acc"])
log_df.to_csv(f"{OUTPUT_DIR}/training_log.csv", index=False)

# ================= SAVE DETAILED CLASSIFICATION REPORT =================
from sklearn.metrics import classification_report
report = classification_report(test_g, test_p, target_names=["Negative", "Positive"], output_dict=True)
pd.DataFrame(report).transpose().to_csv(f"{OUTPUT_DIR}/classification_report.csv")

# ================= SAVE PREDICTIONS (Lebih Lengkap untuk Analisis) =================
pred_df = pd.DataFrame({
    "text": test_texts,        # Tambahkan ini agar bisa dibaca manusia
    "actual": test_g,
    "predicted": test_p,
    "probability": test_prob
})
pred_df.to_csv(f"{OUTPUT_DIR}/test_predictions.csv", index=False)

<>:120: SyntaxWarning: invalid escape sequence '\T'
<>:120: SyntaxWarning: invalid escape sequence '\T'
/tmp/ipykernel_796/2149017565.py:120: SyntaxWarning: invalid escape sequence '\T'
  train_idx = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\train.npy")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LR = 2.88e-06


Epoch 1 | train_loss 0.5734 | val_loss 0.4648 | train_acc 0.6944 | val_acc 0.7812


...

Epoch 33 | train_loss 0.1670 | val_loss 0.4175 | train_acc 0.9368 | val_acc 0.8638


Epoch 34 | train_loss 0.1669 | val_loss 0.4164 | train_acc 0.9360 | val_acc 0.8628


Epoch 35 | train_loss 0.1640 | val_loss 0.4199 | train_acc 0.9366 | val_acc 0.8622
Early stopping triggered at epoch 35



TEST RESULT
0.854 0.854 0.854 0.854


<Figure size 600x600 with 0 Axes>

# **5**

In [5]:
# -*- coding: utf-8 -*-
"""
Created on Mon Mar  2 12:08:39 2026

LR SESUAI LR FINDER

@author: indri
"""

# =========================================================
# CLEAN RESEARCH PIPELINE — DISTILBERT BASELINE
# =========================================================

import os, json, random, time, csv
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch import amp
from torch.optim.lr_scheduler import ReduceLROnPlateau
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
from tqdm import tqdm


# ================= CONFIG =================
CONFIG = {
    "model_name":"distilbert-base-uncased",
    "max_len":128,
    "batch_size":32,
    "epochs":50,
    "lr":2.88e-06,
    "patience":4,
    "seed":42,
    "num_labels":2
}
OUTPUT_DIR = rf"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/DISTILBERT_BASELINE/5.R.L.{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(OUTPUT_DIR, exist_ok=True)
json.dump(CONFIG, open(f"{OUTPUT_DIR}/config.json","w"), indent=4)

DATA_PATH = r"/content/drive/MyDrive/Colab Notebooks/PREPROCESS/5.R.L.csv"

# ================= REPRO =================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ================= AUGMENTASI =================
def text_augment(text):
    tokens = text.split()
    if len(tokens) < 5: return text # Jangan ganggu kalimat pendek

    r = random.random()
    # Random Swap
    if r < 0.15:
        idx1, idx2 = random.sample(range(len(tokens)), 2)
        tokens[idx1], tokens[idx2] = tokens[idx2], tokens[idx1]
    # Random Delete
    elif r < 0.30:
        tokens = [t for t in tokens if random.random() > 0.1]

    return " ".join(tokens)
# ================= TOKENIZER =================
tokenizer = DistilBertTokenizer.from_pretrained(CONFIG["model_name"])

# ================= DATASET =================
class TextDataset(Dataset):
    def __init__(self, texts, labels, augment=False):
        self.texts = texts
        self.labels = labels
        self.augment = augment # Tambahkan flag ini

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        # Jalankan augmentasi hanya jika flag Aktif
        if self.augment:
            text = text_augment(text)

        enc = tokenizer(
            text, # Teks yang sudah (mungkin) diaugmentasi
            truncation=True,
            padding="max_length",
            max_length=CONFIG["max_len"],
            return_tensors="pt"
        )

        return (
            enc["input_ids"].squeeze(0),
            enc["attention_mask"].squeeze(0),
            torch.tensor(self.labels[idx], dtype=torch.long)
        )

# ================= LOAD DATA =================
df = pd.read_csv(DATA_PATH).dropna()
"""
df["sentiment"] = (
    df["sentiment"]
    .astype(str).str.strip().str.lower()
    .replace({"0":"negative","1":"positive"})
    .map({"negative":0,"positive":1})
)

texts = df["review"].astype(str).tolist()
labels = df["sentiment"].tolist()

train_idx = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\train.npy")
val_idx   = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\val.npy")
test_idx  = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\test.npy")

def subset(arr, ids): return [arr[i] for i in ids]
"""
all_reviews = df['review'].astype(str).values
all_labels = df["sentiment"].replace({"0":0, "1":1}).values # Pastikan integer

train_idx = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/train.npy")
val_idx   = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/val.npy")
test_idx  = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/test.npy")

# 2. Distribusikan berdasarkan index yang sudah di-load (Anti-Leakage)
train_texts = [all_reviews[i] for i in train_idx]
train_labels = [all_labels[i] for i in train_idx]

val_texts = [all_reviews[i] for i in val_idx]
val_labels = [all_labels[i] for i in val_idx]

test_texts = [all_reviews[i] for i in test_idx]
test_labels = [all_labels[i] for i in test_idx]

# 3. Verifikasi Kebocoran (Tambahkan ini untuk bukti di Tesis)
assert len(set(train_idx) & set(test_idx)) == 0, "LEAKAGE DETECTED: Train & Test overlap!"
assert len(set(train_idx) & set(val_idx)) == 0, "LEAKAGE DETECTED: Train & Val overlap!"

#==============================================
#KALAU MAU PAKE SAMPLE
#==============================================

DEBUG = False
DEBUG_RATIO = 0.05

if DEBUG:
    train_idx = train_idx[:int(len(train_idx)*DEBUG_RATIO)]
    val_idx   = val_idx[:int(len(val_idx)*DEBUG_RATIO)]
    test_idx  = test_idx[:int(len(test_idx)*DEBUG_RATIO)]

#===============================================
# HANYA train_ds yang boleh augmentasi
"""
train_ds = TextDataset(subset(texts,train_idx), subset(labels,train_idx), augment=True)

# val dan test HARUS augment=False (default)
val_ds   = TextDataset(subset(texts,val_idx),   subset(labels,val_idx), augment=False)
test_ds  = TextDataset(subset(texts,test_idx),  subset(labels,test_idx), augment=False)

train_loader = DataLoader(train_ds,batch_size=CONFIG["batch_size"],shuffle=True)
val_loader   = DataLoader(val_ds,batch_size=CONFIG["batch_size"])
test_loader  = DataLoader(test_ds,batch_size=CONFIG["batch_size"])
"""
# 3. Distribusikan data (Gunakan List Comprehension seperti di BiGRU)
train_texts = [all_reviews[i] for i in train_idx]
train_labels = [all_labels[i] for i in train_idx]

val_texts = [all_reviews[i] for i in val_idx]
val_labels = [all_labels[i] for i in val_idx]

test_texts = [all_reviews[i] for i in test_idx]
test_labels = [all_labels[i] for i in test_idx]

# 4. Inisialisasi Dataset yang BENAR (Hapus vocab, samakan nama class)
train_dataset = TextDataset(train_texts, train_labels, augment=True)
val_dataset   = TextDataset(val_texts, val_labels, augment=False)
test_dataset  = TextDataset(test_texts, test_labels, augment=False)

g = torch.Generator()
g.manual_seed(CONFIG["seed"])

def seed_worker(worker_id):
    worker_seed = (CONFIG["seed"]) + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

train_loader = DataLoader(train_dataset,
                          batch_size=(CONFIG["batch_size"]),
                          shuffle=True,
                          worker_init_fn=seed_worker,
                          generator=g)
val_loader = DataLoader(val_dataset, batch_size=(CONFIG["batch_size"]))
test_loader = DataLoader(test_dataset, batch_size=(CONFIG["batch_size"]))

# ================= MODEL =================
class DistilBERTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained(CONFIG["model_name"])
        self.fc = nn.Linear(768, CONFIG["num_labels"])

    def forward(self, input_ids, attn):
        out = self.bert(input_ids=input_ids, attention_mask=attn)
        cls = out.last_hidden_state[:,0]      # CLS pooling (standard baseline)
        return self.fc(cls)

model = DistilBERTClassifier().to(device)

# ================= OPTIM =================
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=1e-4)
current_lr = optimizer.param_groups[0]["lr"]
print(f"LR = {current_lr:.2e}")
scheduler = ReduceLROnPlateau(optimizer, mode="max", patience=2, factor=0.5)
criterion = nn.CrossEntropyLoss()
scaler = amp.GradScaler(enabled=torch.cuda.is_available())

# ================= TRAIN =================
def train_epoch():
    model.train()
    total_loss, preds, gold = 0, [], []

    for ids,attn,labels in tqdm(train_loader, desc="Training", leave=False):
        ids,attn,labels = ids.to(device), attn.to(device), labels.to(device)

        optimizer.zero_grad()

        with amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
            logits = model(ids,attn)
            loss = criterion(logits,labels)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        preds.extend(torch.argmax(logits,1).cpu().numpy())
        gold.extend(labels.cpu().numpy())

    return total_loss/len(train_loader), preds, gold

# ================= EVAL =================
def evaluate(loader):
    model.eval()
    total_loss, preds, gold, probs = 0, [], [], []

    with torch.no_grad():
        for ids,attn,labels in tqdm(loader, desc="Evaluating", leave=False):
            ids,attn,labels = ids.to(device), attn.to(device), labels.to(device)

            logits = model(ids,attn)
            loss = criterion(logits,labels)

            total_loss += loss.item()

            p = torch.softmax(logits,1)
            preds.extend(torch.argmax(p,1).cpu().numpy())
            gold.extend(labels.cpu().numpy())
            probs.extend(p[:,1].cpu().numpy())

    return total_loss/len(loader), preds, gold, probs


CHECKPOINT_PATH = f"{OUTPUT_DIR}/checkpoint.pt"

start_epoch = 0
best_acc = 0

if os.path.exists(CHECKPOINT_PATH):
    print("Loading checkpoint...")
    ckpt = torch.load(CHECKPOINT_PATH)

    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])

    start_epoch = ckpt["epoch"] + 1
    best_acc = ckpt["best_acc"]

    random.setstate(ckpt["python_rng"])
    np.random.set_state(ckpt["numpy_rng"])
    torch.set_rng_state(ckpt["torch_rng"])
# ================= TRAIN LOOP =================
best_acc = 0
patience = 0
history = []
total_start = time.time()
train_start = time.time()
for epoch in range(start_epoch, CONFIG["epochs"]):

    tr_loss,tr_p,tr_g = train_epoch()
    val_loss,val_p,val_g,_ = evaluate(val_loader)

    #print(f"Epoch {epoch+1} | train {tr_loss:.4f} | val {val_loss:.4f}")
    tr_acc = accuracy_score(tr_g, tr_p)
    val_acc = accuracy_score(val_g, val_p)
    history.append([epoch+1,tr_loss,val_loss,tr_acc,val_acc])

    scheduler.step(val_acc)

    print(f"Epoch {epoch+1} | "
          f"train_loss {tr_loss:.4f} | val_loss {val_loss:.4f} | "
          f"train_acc {tr_acc:.4f} | val_acc {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        patience = 0
        torch.save(model.state_dict(), f"{OUTPUT_DIR}/best.pt")
    else:
        patience += 1
        if patience >= CONFIG["patience"]:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break # INI YANG KURANG!
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "best_acc": best_acc,
        "python_rng": random.getstate(),
        "numpy_rng": np.random.get_state(),
        "torch_rng": torch.get_rng_state()
    }, CHECKPOINT_PATH)

train_time = time.time() - train_start
# ================= TEST =================
test_start = time.time()
model.load_state_dict(torch.load(f"{OUTPUT_DIR}/best.pt"))

test_loss,test_p,test_g,test_prob = evaluate(test_loader)

acc  = accuracy_score(test_g,test_p)
prec = precision_score(test_g,test_p)
rec  = recall_score(test_g,test_p)
f1   = f1_score(test_g,test_p)

print("\nTEST RESULT")
print(acc,prec,rec,f1)
test_time = time.time() - test_start
total_time = time.time() - total_start
# ================= TEST =================
# ... (kode evaluasi Anda)

# 1. HITUNG WAKTU SEBELUM SIMPAN
inference_time_total = test_time
num_samples = len(test_texts)
avg_inference_per_sample = (inference_time_total / num_samples) * 1000

# 2. SIMPAN METRICS (SATU KALI SAJA - JANGAN DOUBLE)
metrics_data = {
    "accuracy": acc,
    "precision": prec,
    "recall": rec,
    "f1": f1,
    "loss": test_loss,
    "time": {
        "train_time_sec": train_time,
        "test_time_sec": test_time,
        "avg_inference_ms": avg_inference_per_sample,
        "total_time_sec": total_time
    }
}

with open(f"{OUTPUT_DIR}/metrics.json", "w") as f:
    json.dump(metrics_data, f, indent=4)

# HAPUS BLOK "# ================= SAVE METRICS =================" BERIKUTNYA


# ================= CURVES =================
epochs=[x[0] for x in history]

plt.plot(epochs,[x[1] for x in history],label="train")
plt.plot(epochs,[x[2] for x in history],label="val")
plt.legend(); plt.grid()
plt.savefig(f"{OUTPUT_DIR}/loss_curve.png")
plt.close()

plt.plot(epochs,[x[3] for x in history],label="train acc")
plt.plot(epochs,[x[4] for x in history],label="val acc")
plt.legend(); plt.grid()
plt.savefig(f"{OUTPUT_DIR}/accuracy_curve.png")
plt.close()


# ================= ROC =================
fpr,tpr,_ = roc_curve(test_g,test_prob)
roc_auc = auc(fpr,tpr)

plt.plot(fpr,tpr,label=f"AUC={roc_auc:.4f}")
plt.plot([0,1],[0,1],'--')
plt.legend(); plt.grid()
plt.savefig(f"{OUTPUT_DIR}/roc.png")
plt.close()


# ================= CM =================
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(test_g, test_p)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative","Positive"]
)

plt.figure(figsize=(6,6))
disp.plot(values_format="d")
plt.title("Confusion Matrix")
plt.grid(False)
plt.savefig(f"{OUTPUT_DIR}/confusion_matrix.png")
plt.close()

# ================= SAVE TRAINING LOGS (CSV) =================
log_df = pd.DataFrame(history, columns=["epoch", "train_loss", "val_loss", "train_acc", "val_acc"])
log_df.to_csv(f"{OUTPUT_DIR}/training_log.csv", index=False)

# ================= SAVE DETAILED CLASSIFICATION REPORT =================
from sklearn.metrics import classification_report
report = classification_report(test_g, test_p, target_names=["Negative", "Positive"], output_dict=True)
pd.DataFrame(report).transpose().to_csv(f"{OUTPUT_DIR}/classification_report.csv")

# ================= SAVE PREDICTIONS (Lebih Lengkap untuk Analisis) =================
pred_df = pd.DataFrame({
    "text": test_texts,        # Tambahkan ini agar bisa dibaca manusia
    "actual": test_g,
    "predicted": test_p,
    "probability": test_prob
})
pred_df.to_csv(f"{OUTPUT_DIR}/test_predictions.csv", index=False)

<>:120: SyntaxWarning: invalid escape sequence '\T'
<>:120: SyntaxWarning: invalid escape sequence '\T'
/tmp/ipykernel_796/1967862422.py:120: SyntaxWarning: invalid escape sequence '\T'
  train_idx = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\train.npy")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LR = 2.88e-06


Epoch 1 | train_loss 0.4985 | val_loss 0.3727 | train_acc 0.7573 | val_acc 0.8370


...

Epoch 17 | train_loss 0.1944 | val_loss 0.3108 | train_acc 0.9229 | val_acc 0.8804


Epoch 18 | train_loss 0.1880 | val_loss 0.3166 | train_acc 0.9269 | val_acc 0.8792


Epoch 19 | train_loss 0.1799 | val_loss 0.3232 | train_acc 0.9318 | val_acc 0.8800
Early stopping triggered at epoch 19



TEST RESULT
0.8796 0.8672600619195047 0.8964 0.8815892997639654


<Figure size 600x600 with 0 Axes>

# **6**

In [6]:
# -*- coding: utf-8 -*-
"""
Created on Mon Mar  2 12:08:39 2026

LR SESUAI LR FINDER

@author: indri
"""

# =========================================================
# CLEAN RESEARCH PIPELINE — DISTILBERT BASELINE
# =========================================================

import os, json, random, time, csv
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch import amp
from torch.optim.lr_scheduler import ReduceLROnPlateau
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
from tqdm import tqdm


# ================= CONFIG =================
CONFIG = {
    "model_name":"distilbert-base-uncased",
    "max_len":128,
    "batch_size":32,
    "epochs":50,
    "lr":2.88e-06,
    "patience":4,
    "seed":42,
    "num_labels":2
}
OUTPUT_DIR = rf"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/DISTILBERT_BASELINE/6.R.S.{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(OUTPUT_DIR, exist_ok=True)
json.dump(CONFIG, open(f"{OUTPUT_DIR}/config.json","w"), indent=4)

DATA_PATH = r"/content/drive/MyDrive/Colab Notebooks/PREPROCESS/6.R.S.csv"

# ================= REPRO =================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ================= AUGMENTASI =================
def text_augment(text):
    tokens = text.split()
    if len(tokens) < 5: return text # Jangan ganggu kalimat pendek

    r = random.random()
    # Random Swap
    if r < 0.15:
        idx1, idx2 = random.sample(range(len(tokens)), 2)
        tokens[idx1], tokens[idx2] = tokens[idx2], tokens[idx1]
    # Random Delete
    elif r < 0.30:
        tokens = [t for t in tokens if random.random() > 0.1]

    return " ".join(tokens)
# ================= TOKENIZER =================
tokenizer = DistilBertTokenizer.from_pretrained(CONFIG["model_name"])

# ================= DATASET =================
class TextDataset(Dataset):
    def __init__(self, texts, labels, augment=False):
        self.texts = texts
        self.labels = labels
        self.augment = augment # Tambahkan flag ini

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        # Jalankan augmentasi hanya jika flag Aktif
        if self.augment:
            text = text_augment(text)

        enc = tokenizer(
            text, # Teks yang sudah (mungkin) diaugmentasi
            truncation=True,
            padding="max_length",
            max_length=CONFIG["max_len"],
            return_tensors="pt"
        )

        return (
            enc["input_ids"].squeeze(0),
            enc["attention_mask"].squeeze(0),
            torch.tensor(self.labels[idx], dtype=torch.long)
        )

# ================= LOAD DATA =================
df = pd.read_csv(DATA_PATH).dropna()
"""
df["sentiment"] = (
    df["sentiment"]
    .astype(str).str.strip().str.lower()
    .replace({"0":"negative","1":"positive"})
    .map({"negative":0,"positive":1})
)

texts = df["review"].astype(str).tolist()
labels = df["sentiment"].tolist()

train_idx = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\train.npy")
val_idx   = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\val.npy")
test_idx  = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\test.npy")

def subset(arr, ids): return [arr[i] for i in ids]
"""
all_reviews = df['review'].astype(str).values
all_labels = df["sentiment"].replace({"0":0, "1":1}).values # Pastikan integer

train_idx = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/train.npy")
val_idx   = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/val.npy")
test_idx  = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/test.npy")

# 2. Distribusikan berdasarkan index yang sudah di-load (Anti-Leakage)
train_texts = [all_reviews[i] for i in train_idx]
train_labels = [all_labels[i] for i in train_idx]

val_texts = [all_reviews[i] for i in val_idx]
val_labels = [all_labels[i] for i in val_idx]

test_texts = [all_reviews[i] for i in test_idx]
test_labels = [all_labels[i] for i in test_idx]

# 3. Verifikasi Kebocoran (Tambahkan ini untuk bukti di Tesis)
assert len(set(train_idx) & set(test_idx)) == 0, "LEAKAGE DETECTED: Train & Test overlap!"
assert len(set(train_idx) & set(val_idx)) == 0, "LEAKAGE DETECTED: Train & Val overlap!"

#==============================================
#KALAU MAU PAKE SAMPLE
#==============================================

DEBUG = False
DEBUG_RATIO = 0.05

if DEBUG:
    train_idx = train_idx[:int(len(train_idx)*DEBUG_RATIO)]
    val_idx   = val_idx[:int(len(val_idx)*DEBUG_RATIO)]
    test_idx  = test_idx[:int(len(test_idx)*DEBUG_RATIO)]

#===============================================
# HANYA train_ds yang boleh augmentasi
"""
train_ds = TextDataset(subset(texts,train_idx), subset(labels,train_idx), augment=True)

# val dan test HARUS augment=False (default)
val_ds   = TextDataset(subset(texts,val_idx),   subset(labels,val_idx), augment=False)
test_ds  = TextDataset(subset(texts,test_idx),  subset(labels,test_idx), augment=False)

train_loader = DataLoader(train_ds,batch_size=CONFIG["batch_size"],shuffle=True)
val_loader   = DataLoader(val_ds,batch_size=CONFIG["batch_size"])
test_loader  = DataLoader(test_ds,batch_size=CONFIG["batch_size"])
"""
# 3. Distribusikan data (Gunakan List Comprehension seperti di BiGRU)
train_texts = [all_reviews[i] for i in train_idx]
train_labels = [all_labels[i] for i in train_idx]

val_texts = [all_reviews[i] for i in val_idx]
val_labels = [all_labels[i] for i in val_idx]

test_texts = [all_reviews[i] for i in test_idx]
test_labels = [all_labels[i] for i in test_idx]

# 4. Inisialisasi Dataset yang BENAR (Hapus vocab, samakan nama class)
train_dataset = TextDataset(train_texts, train_labels, augment=True)
val_dataset   = TextDataset(val_texts, val_labels, augment=False)
test_dataset  = TextDataset(test_texts, test_labels, augment=False)

g = torch.Generator()
g.manual_seed(CONFIG["seed"])

def seed_worker(worker_id):
    worker_seed = (CONFIG["seed"]) + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

train_loader = DataLoader(train_dataset,
                          batch_size=(CONFIG["batch_size"]),
                          shuffle=True,
                          worker_init_fn=seed_worker,
                          generator=g)
val_loader = DataLoader(val_dataset, batch_size=(CONFIG["batch_size"]))
test_loader = DataLoader(test_dataset, batch_size=(CONFIG["batch_size"]))

# ================= MODEL =================
class DistilBERTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained(CONFIG["model_name"])
        self.fc = nn.Linear(768, CONFIG["num_labels"])

    def forward(self, input_ids, attn):
        out = self.bert(input_ids=input_ids, attention_mask=attn)
        cls = out.last_hidden_state[:,0]      # CLS pooling (standard baseline)
        return self.fc(cls)

model = DistilBERTClassifier().to(device)

# ================= OPTIM =================
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=1e-4)
current_lr = optimizer.param_groups[0]["lr"]
print(f"LR = {current_lr:.2e}")
scheduler = ReduceLROnPlateau(optimizer, mode="max", patience=2, factor=0.5)
criterion = nn.CrossEntropyLoss()
scaler = amp.GradScaler(enabled=torch.cuda.is_available())

# ================= TRAIN =================
def train_epoch():
    model.train()
    total_loss, preds, gold = 0, [], []

    for ids,attn,labels in tqdm(train_loader, desc="Training", leave=False):
        ids,attn,labels = ids.to(device), attn.to(device), labels.to(device)

        optimizer.zero_grad()

        with amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
            logits = model(ids,attn)
            loss = criterion(logits,labels)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        preds.extend(torch.argmax(logits,1).cpu().numpy())
        gold.extend(labels.cpu().numpy())

    return total_loss/len(train_loader), preds, gold

# ================= EVAL =================
def evaluate(loader):
    model.eval()
    total_loss, preds, gold, probs = 0, [], [], []

    with torch.no_grad():
        for ids,attn,labels in tqdm(loader, desc="Evaluating", leave=False):
            ids,attn,labels = ids.to(device), attn.to(device), labels.to(device)

            logits = model(ids,attn)
            loss = criterion(logits,labels)

            total_loss += loss.item()

            p = torch.softmax(logits,1)
            preds.extend(torch.argmax(p,1).cpu().numpy())
            gold.extend(labels.cpu().numpy())
            probs.extend(p[:,1].cpu().numpy())

    return total_loss/len(loader), preds, gold, probs


CHECKPOINT_PATH = f"{OUTPUT_DIR}/checkpoint.pt"

start_epoch = 0
best_acc = 0

if os.path.exists(CHECKPOINT_PATH):
    print("Loading checkpoint...")
    ckpt = torch.load(CHECKPOINT_PATH)

    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])

    start_epoch = ckpt["epoch"] + 1
    best_acc = ckpt["best_acc"]

    random.setstate(ckpt["python_rng"])
    np.random.set_state(ckpt["numpy_rng"])
    torch.set_rng_state(ckpt["torch_rng"])
# ================= TRAIN LOOP =================
best_acc = 0
patience = 0
history = []
total_start = time.time()
train_start = time.time()
for epoch in range(start_epoch, CONFIG["epochs"]):

    tr_loss,tr_p,tr_g = train_epoch()
    val_loss,val_p,val_g,_ = evaluate(val_loader)

    #print(f"Epoch {epoch+1} | train {tr_loss:.4f} | val {val_loss:.4f}")
    tr_acc = accuracy_score(tr_g, tr_p)
    val_acc = accuracy_score(val_g, val_p)
    history.append([epoch+1,tr_loss,val_loss,tr_acc,val_acc])

    scheduler.step(val_acc)

    print(f"Epoch {epoch+1} | "
          f"train_loss {tr_loss:.4f} | val_loss {val_loss:.4f} | "
          f"train_acc {tr_acc:.4f} | val_acc {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        patience = 0
        torch.save(model.state_dict(), f"{OUTPUT_DIR}/best.pt")
    else:
        patience += 1
        if patience >= CONFIG["patience"]:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break # INI YANG KURANG!
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "best_acc": best_acc,
        "python_rng": random.getstate(),
        "numpy_rng": np.random.get_state(),
        "torch_rng": torch.get_rng_state()
    }, CHECKPOINT_PATH)

train_time = time.time() - train_start
# ================= TEST =================
test_start = time.time()
model.load_state_dict(torch.load(f"{OUTPUT_DIR}/best.pt"))

test_loss,test_p,test_g,test_prob = evaluate(test_loader)

acc  = accuracy_score(test_g,test_p)
prec = precision_score(test_g,test_p)
rec  = recall_score(test_g,test_p)
f1   = f1_score(test_g,test_p)

print("\nTEST RESULT")
print(acc,prec,rec,f1)
test_time = time.time() - test_start
total_time = time.time() - total_start
# ================= TEST =================
# ... (kode evaluasi Anda)

# 1. HITUNG WAKTU SEBELUM SIMPAN
inference_time_total = test_time
num_samples = len(test_texts)
avg_inference_per_sample = (inference_time_total / num_samples) * 1000

# 2. SIMPAN METRICS (SATU KALI SAJA - JANGAN DOUBLE)
metrics_data = {
    "accuracy": acc,
    "precision": prec,
    "recall": rec,
    "f1": f1,
    "loss": test_loss,
    "time": {
        "train_time_sec": train_time,
        "test_time_sec": test_time,
        "avg_inference_ms": avg_inference_per_sample,
        "total_time_sec": total_time
    }
}

with open(f"{OUTPUT_DIR}/metrics.json", "w") as f:
    json.dump(metrics_data, f, indent=4)

# HAPUS BLOK "# ================= SAVE METRICS =================" BERIKUTNYA


# ================= CURVES =================
epochs=[x[0] for x in history]

plt.plot(epochs,[x[1] for x in history],label="train")
plt.plot(epochs,[x[2] for x in history],label="val")
plt.legend(); plt.grid()
plt.savefig(f"{OUTPUT_DIR}/loss_curve.png")
plt.close()

plt.plot(epochs,[x[3] for x in history],label="train acc")
plt.plot(epochs,[x[4] for x in history],label="val acc")
plt.legend(); plt.grid()
plt.savefig(f"{OUTPUT_DIR}/accuracy_curve.png")
plt.close()


# ================= ROC =================
fpr,tpr,_ = roc_curve(test_g,test_prob)
roc_auc = auc(fpr,tpr)

plt.plot(fpr,tpr,label=f"AUC={roc_auc:.4f}")
plt.plot([0,1],[0,1],'--')
plt.legend(); plt.grid()
plt.savefig(f"{OUTPUT_DIR}/roc.png")
plt.close()


# ================= CM =================
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(test_g, test_p)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative","Positive"]
)

plt.figure(figsize=(6,6))
disp.plot(values_format="d")
plt.title("Confusion Matrix")
plt.grid(False)
plt.savefig(f"{OUTPUT_DIR}/confusion_matrix.png")
plt.close()

# ================= SAVE TRAINING LOGS (CSV) =================
log_df = pd.DataFrame(history, columns=["epoch", "train_loss", "val_loss", "train_acc", "val_acc"])
log_df.to_csv(f"{OUTPUT_DIR}/training_log.csv", index=False)

# ================= SAVE DETAILED CLASSIFICATION REPORT =================
from sklearn.metrics import classification_report
report = classification_report(test_g, test_p, target_names=["Negative", "Positive"], output_dict=True)
pd.DataFrame(report).transpose().to_csv(f"{OUTPUT_DIR}/classification_report.csv")

# ================= SAVE PREDICTIONS (Lebih Lengkap untuk Analisis) =================
pred_df = pd.DataFrame({
    "text": test_texts,        # Tambahkan ini agar bisa dibaca manusia
    "actual": test_g,
    "predicted": test_p,
    "probability": test_prob
})
pred_df.to_csv(f"{OUTPUT_DIR}/test_predictions.csv", index=False)

<>:120: SyntaxWarning: invalid escape sequence '\T'
<>:120: SyntaxWarning: invalid escape sequence '\T'
/tmp/ipykernel_796/2488161802.py:120: SyntaxWarning: invalid escape sequence '\T'
  train_idx = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\train.npy")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LR = 2.88e-06


Epoch 1 | train_loss 0.5771 | val_loss 0.4606 | train_acc 0.6926 | val_acc 0.7818


...

Epoch 22 | train_loss 0.2113 | val_loss 0.3727 | train_acc 0.9160 | val_acc 0.8550


Epoch 23 | train_loss 0.2085 | val_loss 0.3826 | train_acc 0.9176 | val_acc 0.8548


Epoch 24 | train_loss 0.1925 | val_loss 0.3882 | train_acc 0.9244 | val_acc 0.8568
Early stopping triggered at epoch 24



TEST RESULT
0.862 0.8443683409436834 0.8876 0.8654446177847114


<Figure size 600x600 with 0 Axes>

# **7**

In [1]:
# -*- coding: utf-8 -*-
"""
Created on Mon Mar  2 12:08:39 2026

LR SESUAI LR FINDER

@author: indri
"""

# =========================================================
# CLEAN RESEARCH PIPELINE — DISTILBERT BASELINE
# =========================================================

import os, json, random, time, csv
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch import amp
from torch.optim.lr_scheduler import ReduceLROnPlateau
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
from tqdm import tqdm


# ================= CONFIG =================
CONFIG = {
    "model_name":"distilbert-base-uncased",
    "max_len":128,
    "batch_size":32,
    "epochs":50,
    "lr":2.88e-06,
    "patience":4,
    "seed":42,
    "num_labels":2
}
OUTPUT_DIR = rf"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/DISTILBERT_BASELINE/7.L.S.{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(OUTPUT_DIR, exist_ok=True)
json.dump(CONFIG, open(f"{OUTPUT_DIR}/config.json","w"), indent=4)

DATA_PATH = r"/content/drive/MyDrive/Colab Notebooks/PREPROCESS/7.L.S.csv"

# ================= REPRO =================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ================= AUGMENTASI =================
def text_augment(text):
    tokens = text.split()
    if len(tokens) < 5: return text # Jangan ganggu kalimat pendek

    r = random.random()
    # Random Swap
    if r < 0.15:
        idx1, idx2 = random.sample(range(len(tokens)), 2)
        tokens[idx1], tokens[idx2] = tokens[idx2], tokens[idx1]
    # Random Delete
    elif r < 0.30:
        tokens = [t for t in tokens if random.random() > 0.1]

    return " ".join(tokens)
# ================= TOKENIZER =================
tokenizer = DistilBertTokenizer.from_pretrained(CONFIG["model_name"])

# ================= DATASET =================
class TextDataset(Dataset):
    def __init__(self, texts, labels, augment=False):
        self.texts = texts
        self.labels = labels
        self.augment = augment # Tambahkan flag ini

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        # Jalankan augmentasi hanya jika flag Aktif
        if self.augment:
            text = text_augment(text)

        enc = tokenizer(
            text, # Teks yang sudah (mungkin) diaugmentasi
            truncation=True,
            padding="max_length",
            max_length=CONFIG["max_len"],
            return_tensors="pt"
        )

        return (
            enc["input_ids"].squeeze(0),
            enc["attention_mask"].squeeze(0),
            torch.tensor(self.labels[idx], dtype=torch.long)
        )

# ================= LOAD DATA =================
df = pd.read_csv(DATA_PATH).dropna()
"""
df["sentiment"] = (
    df["sentiment"]
    .astype(str).str.strip().str.lower()
    .replace({"0":"negative","1":"positive"})
    .map({"negative":0,"positive":1})
)

texts = df["review"].astype(str).tolist()
labels = df["sentiment"].tolist()

train_idx = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\train.npy")
val_idx   = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\val.npy")
test_idx  = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\test.npy")

def subset(arr, ids): return [arr[i] for i in ids]
"""
all_reviews = df['review'].astype(str).values
all_labels = df["sentiment"].replace({"0":0, "1":1}).values # Pastikan integer

train_idx = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/train.npy")
val_idx   = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/val.npy")
test_idx  = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/test.npy")

# 2. Distribusikan berdasarkan index yang sudah di-load (Anti-Leakage)
train_texts = [all_reviews[i] for i in train_idx]
train_labels = [all_labels[i] for i in train_idx]

val_texts = [all_reviews[i] for i in val_idx]
val_labels = [all_labels[i] for i in val_idx]

test_texts = [all_reviews[i] for i in test_idx]
test_labels = [all_labels[i] for i in test_idx]

# 3. Verifikasi Kebocoran (Tambahkan ini untuk bukti di Tesis)
assert len(set(train_idx) & set(test_idx)) == 0, "LEAKAGE DETECTED: Train & Test overlap!"
assert len(set(train_idx) & set(val_idx)) == 0, "LEAKAGE DETECTED: Train & Val overlap!"

#==============================================
#KALAU MAU PAKE SAMPLE
#==============================================

DEBUG = False
DEBUG_RATIO = 0.05

if DEBUG:
    train_idx = train_idx[:int(len(train_idx)*DEBUG_RATIO)]
    val_idx   = val_idx[:int(len(val_idx)*DEBUG_RATIO)]
    test_idx  = test_idx[:int(len(test_idx)*DEBUG_RATIO)]

#===============================================
# HANYA train_ds yang boleh augmentasi
"""
train_ds = TextDataset(subset(texts,train_idx), subset(labels,train_idx), augment=True)

# val dan test HARUS augment=False (default)
val_ds   = TextDataset(subset(texts,val_idx),   subset(labels,val_idx), augment=False)
test_ds  = TextDataset(subset(texts,test_idx),  subset(labels,test_idx), augment=False)

train_loader = DataLoader(train_ds,batch_size=CONFIG["batch_size"],shuffle=True)
val_loader   = DataLoader(val_ds,batch_size=CONFIG["batch_size"])
test_loader  = DataLoader(test_ds,batch_size=CONFIG["batch_size"])
"""
# 3. Distribusikan data (Gunakan List Comprehension seperti di BiGRU)
train_texts = [all_reviews[i] for i in train_idx]
train_labels = [all_labels[i] for i in train_idx]

val_texts = [all_reviews[i] for i in val_idx]
val_labels = [all_labels[i] for i in val_idx]

test_texts = [all_reviews[i] for i in test_idx]
test_labels = [all_labels[i] for i in test_idx]

# 4. Inisialisasi Dataset yang BENAR (Hapus vocab, samakan nama class)
train_dataset = TextDataset(train_texts, train_labels, augment=True)
val_dataset   = TextDataset(val_texts, val_labels, augment=False)
test_dataset  = TextDataset(test_texts, test_labels, augment=False)

g = torch.Generator()
g.manual_seed(CONFIG["seed"])

def seed_worker(worker_id):
    worker_seed = (CONFIG["seed"]) + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

train_loader = DataLoader(train_dataset,
                          batch_size=(CONFIG["batch_size"]),
                          shuffle=True,
                          worker_init_fn=seed_worker,
                          generator=g)
val_loader = DataLoader(val_dataset, batch_size=(CONFIG["batch_size"]))
test_loader = DataLoader(test_dataset, batch_size=(CONFIG["batch_size"]))

# ================= MODEL =================
class DistilBERTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained(CONFIG["model_name"])
        self.fc = nn.Linear(768, CONFIG["num_labels"])

    def forward(self, input_ids, attn):
        out = self.bert(input_ids=input_ids, attention_mask=attn)
        cls = out.last_hidden_state[:,0]      # CLS pooling (standard baseline)
        return self.fc(cls)

model = DistilBERTClassifier().to(device)

# ================= OPTIM =================
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=1e-4)
current_lr = optimizer.param_groups[0]["lr"]
print(f"LR = {current_lr:.2e}")
scheduler = ReduceLROnPlateau(optimizer, mode="max", patience=2, factor=0.5)
criterion = nn.CrossEntropyLoss()
scaler = amp.GradScaler(enabled=torch.cuda.is_available())

# ================= TRAIN =================
def train_epoch():
    model.train()
    total_loss, preds, gold = 0, [], []

    for ids,attn,labels in tqdm(train_loader, desc="Training", leave=False):
        ids,attn,labels = ids.to(device), attn.to(device), labels.to(device)

        optimizer.zero_grad()

        with amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
            logits = model(ids,attn)
            loss = criterion(logits,labels)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        preds.extend(torch.argmax(logits,1).cpu().numpy())
        gold.extend(labels.cpu().numpy())

    return total_loss/len(train_loader), preds, gold

# ================= EVAL =================
def evaluate(loader):
    model.eval()
    total_loss, preds, gold, probs = 0, [], [], []

    with torch.no_grad():
        for ids,attn,labels in tqdm(loader, desc="Evaluating", leave=False):
            ids,attn,labels = ids.to(device), attn.to(device), labels.to(device)

            logits = model(ids,attn)
            loss = criterion(logits,labels)

            total_loss += loss.item()

            p = torch.softmax(logits,1)
            preds.extend(torch.argmax(p,1).cpu().numpy())
            gold.extend(labels.cpu().numpy())
            probs.extend(p[:,1].cpu().numpy())

    return total_loss/len(loader), preds, gold, probs


CHECKPOINT_PATH = f"{OUTPUT_DIR}/checkpoint.pt"

start_epoch = 0
best_acc = 0

if os.path.exists(CHECKPOINT_PATH):
    print("Loading checkpoint...")
    ckpt = torch.load(CHECKPOINT_PATH)

    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])

    start_epoch = ckpt["epoch"] + 1
    best_acc = ckpt["best_acc"]

    random.setstate(ckpt["python_rng"])
    np.random.set_state(ckpt["numpy_rng"])
    torch.set_rng_state(ckpt["torch_rng"])
# ================= TRAIN LOOP =================
best_acc = 0
patience = 0
history = []
total_start = time.time()
train_start = time.time()
for epoch in range(start_epoch, CONFIG["epochs"]):

    tr_loss,tr_p,tr_g = train_epoch()
    val_loss,val_p,val_g,_ = evaluate(val_loader)

    #print(f"Epoch {epoch+1} | train {tr_loss:.4f} | val {val_loss:.4f}")
    tr_acc = accuracy_score(tr_g, tr_p)
    val_acc = accuracy_score(val_g, val_p)
    history.append([epoch+1,tr_loss,val_loss,tr_acc,val_acc])

    scheduler.step(val_acc)

    print(f"Epoch {epoch+1} | "
          f"train_loss {tr_loss:.4f} | val_loss {val_loss:.4f} | "
          f"train_acc {tr_acc:.4f} | val_acc {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        patience = 0
        torch.save(model.state_dict(), f"{OUTPUT_DIR}/best.pt")
    else:
        patience += 1
        if patience >= CONFIG["patience"]:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break # INI YANG KURANG!
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "best_acc": best_acc,
        "python_rng": random.getstate(),
        "numpy_rng": np.random.get_state(),
        "torch_rng": torch.get_rng_state()
    }, CHECKPOINT_PATH)

train_time = time.time() - train_start
# ================= TEST =================
test_start = time.time()
model.load_state_dict(torch.load(f"{OUTPUT_DIR}/best.pt"))

test_loss,test_p,test_g,test_prob = evaluate(test_loader)

acc  = accuracy_score(test_g,test_p)
prec = precision_score(test_g,test_p)
rec  = recall_score(test_g,test_p)
f1   = f1_score(test_g,test_p)

print("\nTEST RESULT")
print(acc,prec,rec,f1)
test_time = time.time() - test_start
total_time = time.time() - total_start
# ================= TEST =================
# ... (kode evaluasi Anda)

# 1. HITUNG WAKTU SEBELUM SIMPAN
inference_time_total = test_time
num_samples = len(test_texts)
avg_inference_per_sample = (inference_time_total / num_samples) * 1000

# 2. SIMPAN METRICS (SATU KALI SAJA - JANGAN DOUBLE)
metrics_data = {
    "accuracy": acc,
    "precision": prec,
    "recall": rec,
    "f1": f1,
    "loss": test_loss,
    "time": {
        "train_time_sec": train_time,
        "test_time_sec": test_time,
        "avg_inference_ms": avg_inference_per_sample,
        "total_time_sec": total_time
    }
}

with open(f"{OUTPUT_DIR}/metrics.json", "w") as f:
    json.dump(metrics_data, f, indent=4)

# HAPUS BLOK "# ================= SAVE METRICS =================" BERIKUTNYA


# ================= CURVES =================
epochs=[x[0] for x in history]

plt.plot(epochs,[x[1] for x in history],label="train")
plt.plot(epochs,[x[2] for x in history],label="val")
plt.legend(); plt.grid()
plt.savefig(f"{OUTPUT_DIR}/loss_curve.png")
plt.close()

plt.plot(epochs,[x[3] for x in history],label="train acc")
plt.plot(epochs,[x[4] for x in history],label="val acc")
plt.legend(); plt.grid()
plt.savefig(f"{OUTPUT_DIR}/accuracy_curve.png")
plt.close()


# ================= ROC =================
fpr,tpr,_ = roc_curve(test_g,test_prob)
roc_auc = auc(fpr,tpr)

plt.plot(fpr,tpr,label=f"AUC={roc_auc:.4f}")
plt.plot([0,1],[0,1],'--')
plt.legend(); plt.grid()
plt.savefig(f"{OUTPUT_DIR}/roc.png")
plt.close()


# ================= CM =================
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(test_g, test_p)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative","Positive"]
)

plt.figure(figsize=(6,6))
disp.plot(values_format="d")
plt.title("Confusion Matrix")
plt.grid(False)
plt.savefig(f"{OUTPUT_DIR}/confusion_matrix.png")
plt.close()

# ================= SAVE TRAINING LOGS (CSV) =================
log_df = pd.DataFrame(history, columns=["epoch", "train_loss", "val_loss", "train_acc", "val_acc"])
log_df.to_csv(f"{OUTPUT_DIR}/training_log.csv", index=False)

# ================= SAVE DETAILED CLASSIFICATION REPORT =================
from sklearn.metrics import classification_report
report = classification_report(test_g, test_p, target_names=["Negative", "Positive"], output_dict=True)
pd.DataFrame(report).transpose().to_csv(f"{OUTPUT_DIR}/classification_report.csv")

# ================= SAVE PREDICTIONS (Lebih Lengkap untuk Analisis) =================
pred_df = pd.DataFrame({
    "text": test_texts,        # Tambahkan ini agar bisa dibaca manusia
    "actual": test_g,
    "predicted": test_p,
    "probability": test_prob
})
pred_df.to_csv(f"{OUTPUT_DIR}/test_predictions.csv", index=False)

<>:120: SyntaxWarning: invalid escape sequence '\T'
<>:120: SyntaxWarning: invalid escape sequence '\T'
/tmp/ipykernel_3463/3547034265.py:120: SyntaxWarning: invalid escape sequence '\T'
  train_idx = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\train.npy")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

...

Epoch 21 | train_loss 0.2246 | val_loss 0.3748 | train_acc 0.9101 | val_acc 0.8580


Epoch 22 | train_loss 0.2196 | val_loss 0.3785 | train_acc 0.9119 | val_acc 0.8564


Epoch 23 | train_loss 0.2081 | val_loss 0.3898 | train_acc 0.9174 | val_acc 0.8566
Early stopping triggered at epoch 23



TEST RESULT
0.8542 0.8490342924714229 0.8616 0.8552709946396665


<Figure size 600x600 with 0 Axes>

# **8**

In [2]:
# -*- coding: utf-8 -*-
"""
Created on Mon Mar  2 12:08:39 2026

LR SESUAI LR FINDER

@author: indri
"""

# =========================================================
# CLEAN RESEARCH PIPELINE — DISTILBERT BASELINE
# =========================================================

import os, json, random, time, csv
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch import amp
from torch.optim.lr_scheduler import ReduceLROnPlateau
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
from tqdm import tqdm


# ================= CONFIG =================
CONFIG = {
    "model_name":"distilbert-base-uncased",
    "max_len":128,
    "batch_size":32,
    "epochs":50,
    "lr":2.88e-06,
    "patience":4,
    "seed":42,
    "num_labels":2
}
OUTPUT_DIR = rf"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/DISTILBERT_BASELINE/8.R.L.S.{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(OUTPUT_DIR, exist_ok=True)
json.dump(CONFIG, open(f"{OUTPUT_DIR}/config.json","w"), indent=4)

DATA_PATH = r"/content/drive/MyDrive/Colab Notebooks/PREPROCESS/8.R.L.S.csv"

# ================= REPRO =================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ================= AUGMENTASI =================
def text_augment(text):
    tokens = text.split()
    if len(tokens) < 5: return text # Jangan ganggu kalimat pendek

    r = random.random()
    # Random Swap
    if r < 0.15:
        idx1, idx2 = random.sample(range(len(tokens)), 2)
        tokens[idx1], tokens[idx2] = tokens[idx2], tokens[idx1]
    # Random Delete
    elif r < 0.30:
        tokens = [t for t in tokens if random.random() > 0.1]

    return " ".join(tokens)
# ================= TOKENIZER =================
tokenizer = DistilBertTokenizer.from_pretrained(CONFIG["model_name"])

# ================= DATASET =================
class TextDataset(Dataset):
    def __init__(self, texts, labels, augment=False):
        self.texts = texts
        self.labels = labels
        self.augment = augment # Tambahkan flag ini

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        # Jalankan augmentasi hanya jika flag Aktif
        if self.augment:
            text = text_augment(text)

        enc = tokenizer(
            text, # Teks yang sudah (mungkin) diaugmentasi
            truncation=True,
            padding="max_length",
            max_length=CONFIG["max_len"],
            return_tensors="pt"
        )

        return (
            enc["input_ids"].squeeze(0),
            enc["attention_mask"].squeeze(0),
            torch.tensor(self.labels[idx], dtype=torch.long)
        )

# ================= LOAD DATA =================
df = pd.read_csv(DATA_PATH).dropna()
"""
df["sentiment"] = (
    df["sentiment"]
    .astype(str).str.strip().str.lower()
    .replace({"0":"negative","1":"positive"})
    .map({"negative":0,"positive":1})
)

texts = df["review"].astype(str).tolist()
labels = df["sentiment"].tolist()

train_idx = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\train.npy")
val_idx   = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\val.npy")
test_idx  = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\test.npy")

def subset(arr, ids): return [arr[i] for i in ids]
"""
all_reviews = df['review'].astype(str).values
all_labels = df["sentiment"].replace({"0":0, "1":1}).values # Pastikan integer

train_idx = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/train.npy")
val_idx   = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/val.npy")
test_idx  = np.load(r"/content/drive/MyDrive/Colab Notebooks/TESIS FIX/SPLIT IDX/test.npy")

# 2. Distribusikan berdasarkan index yang sudah di-load (Anti-Leakage)
train_texts = [all_reviews[i] for i in train_idx]
train_labels = [all_labels[i] for i in train_idx]

val_texts = [all_reviews[i] for i in val_idx]
val_labels = [all_labels[i] for i in val_idx]

test_texts = [all_reviews[i] for i in test_idx]
test_labels = [all_labels[i] for i in test_idx]

# 3. Verifikasi Kebocoran (Tambahkan ini untuk bukti di Tesis)
assert len(set(train_idx) & set(test_idx)) == 0, "LEAKAGE DETECTED: Train & Test overlap!"
assert len(set(train_idx) & set(val_idx)) == 0, "LEAKAGE DETECTED: Train & Val overlap!"

#==============================================
#KALAU MAU PAKE SAMPLE
#==============================================

DEBUG = False
DEBUG_RATIO = 0.05

if DEBUG:
    train_idx = train_idx[:int(len(train_idx)*DEBUG_RATIO)]
    val_idx   = val_idx[:int(len(val_idx)*DEBUG_RATIO)]
    test_idx  = test_idx[:int(len(test_idx)*DEBUG_RATIO)]

#===============================================
# HANYA train_ds yang boleh augmentasi
"""
train_ds = TextDataset(subset(texts,train_idx), subset(labels,train_idx), augment=True)

# val dan test HARUS augment=False (default)
val_ds   = TextDataset(subset(texts,val_idx),   subset(labels,val_idx), augment=False)
test_ds  = TextDataset(subset(texts,test_idx),  subset(labels,test_idx), augment=False)

train_loader = DataLoader(train_ds,batch_size=CONFIG["batch_size"],shuffle=True)
val_loader   = DataLoader(val_ds,batch_size=CONFIG["batch_size"])
test_loader  = DataLoader(test_ds,batch_size=CONFIG["batch_size"])
"""
# 3. Distribusikan data (Gunakan List Comprehension seperti di BiGRU)
train_texts = [all_reviews[i] for i in train_idx]
train_labels = [all_labels[i] for i in train_idx]

val_texts = [all_reviews[i] for i in val_idx]
val_labels = [all_labels[i] for i in val_idx]

test_texts = [all_reviews[i] for i in test_idx]
test_labels = [all_labels[i] for i in test_idx]

# 4. Inisialisasi Dataset yang BENAR (Hapus vocab, samakan nama class)
train_dataset = TextDataset(train_texts, train_labels, augment=True)
val_dataset   = TextDataset(val_texts, val_labels, augment=False)
test_dataset  = TextDataset(test_texts, test_labels, augment=False)

g = torch.Generator()
g.manual_seed(CONFIG["seed"])

def seed_worker(worker_id):
    worker_seed = (CONFIG["seed"]) + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

train_loader = DataLoader(train_dataset,
                          batch_size=(CONFIG["batch_size"]),
                          shuffle=True,
                          worker_init_fn=seed_worker,
                          generator=g)
val_loader = DataLoader(val_dataset, batch_size=(CONFIG["batch_size"]))
test_loader = DataLoader(test_dataset, batch_size=(CONFIG["batch_size"]))

# ================= MODEL =================
class DistilBERTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained(CONFIG["model_name"])
        self.fc = nn.Linear(768, CONFIG["num_labels"])

    def forward(self, input_ids, attn):
        out = self.bert(input_ids=input_ids, attention_mask=attn)
        cls = out.last_hidden_state[:,0]      # CLS pooling (standard baseline)
        return self.fc(cls)

model = DistilBERTClassifier().to(device)

# ================= OPTIM =================
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=1e-4)
current_lr = optimizer.param_groups[0]["lr"]
print(f"LR = {current_lr:.2e}")
scheduler = ReduceLROnPlateau(optimizer, mode="max", patience=2, factor=0.5)
criterion = nn.CrossEntropyLoss()
scaler = amp.GradScaler(enabled=torch.cuda.is_available())

# ================= TRAIN =================
def train_epoch():
    model.train()
    total_loss, preds, gold = 0, [], []

    for ids,attn,labels in tqdm(train_loader, desc="Training", leave=False):
        ids,attn,labels = ids.to(device), attn.to(device), labels.to(device)

        optimizer.zero_grad()

        with amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
            logits = model(ids,attn)
            loss = criterion(logits,labels)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        preds.extend(torch.argmax(logits,1).cpu().numpy())
        gold.extend(labels.cpu().numpy())

    return total_loss/len(train_loader), preds, gold

# ================= EVAL =================
def evaluate(loader):
    model.eval()
    total_loss, preds, gold, probs = 0, [], [], []

    with torch.no_grad():
        for ids,attn,labels in tqdm(loader, desc="Evaluating", leave=False):
            ids,attn,labels = ids.to(device), attn.to(device), labels.to(device)

            logits = model(ids,attn)
            loss = criterion(logits,labels)

            total_loss += loss.item()

            p = torch.softmax(logits,1)
            preds.extend(torch.argmax(p,1).cpu().numpy())
            gold.extend(labels.cpu().numpy())
            probs.extend(p[:,1].cpu().numpy())

    return total_loss/len(loader), preds, gold, probs


CHECKPOINT_PATH = f"{OUTPUT_DIR}/checkpoint.pt"

start_epoch = 0
best_acc = 0

if os.path.exists(CHECKPOINT_PATH):
    print("Loading checkpoint...")
    ckpt = torch.load(CHECKPOINT_PATH)

    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])

    start_epoch = ckpt["epoch"] + 1
    best_acc = ckpt["best_acc"]

    random.setstate(ckpt["python_rng"])
    np.random.set_state(ckpt["numpy_rng"])
    torch.set_rng_state(ckpt["torch_rng"])
# ================= TRAIN LOOP =================
best_acc = 0
patience = 0
history = []
total_start = time.time()
train_start = time.time()
for epoch in range(start_epoch, CONFIG["epochs"]):

    tr_loss,tr_p,tr_g = train_epoch()
    val_loss,val_p,val_g,_ = evaluate(val_loader)

    #print(f"Epoch {epoch+1} | train {tr_loss:.4f} | val {val_loss:.4f}")
    tr_acc = accuracy_score(tr_g, tr_p)
    val_acc = accuracy_score(val_g, val_p)
    history.append([epoch+1,tr_loss,val_loss,tr_acc,val_acc])

    scheduler.step(val_acc)

    print(f"Epoch {epoch+1} | "
          f"train_loss {tr_loss:.4f} | val_loss {val_loss:.4f} | "
          f"train_acc {tr_acc:.4f} | val_acc {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        patience = 0
        torch.save(model.state_dict(), f"{OUTPUT_DIR}/best.pt")
    else:
        patience += 1
        if patience >= CONFIG["patience"]:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break # INI YANG KURANG!
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "best_acc": best_acc,
        "python_rng": random.getstate(),
        "numpy_rng": np.random.get_state(),
        "torch_rng": torch.get_rng_state()
    }, CHECKPOINT_PATH)

train_time = time.time() - train_start
# ================= TEST =================
test_start = time.time()
model.load_state_dict(torch.load(f"{OUTPUT_DIR}/best.pt"))

test_loss,test_p,test_g,test_prob = evaluate(test_loader)

acc  = accuracy_score(test_g,test_p)
prec = precision_score(test_g,test_p)
rec  = recall_score(test_g,test_p)
f1   = f1_score(test_g,test_p)

print("\nTEST RESULT")
print(acc,prec,rec,f1)
test_time = time.time() - test_start
total_time = time.time() - total_start
# ================= TEST =================
# ... (kode evaluasi Anda)

# 1. HITUNG WAKTU SEBELUM SIMPAN
inference_time_total = test_time
num_samples = len(test_texts)
avg_inference_per_sample = (inference_time_total / num_samples) * 1000

# 2. SIMPAN METRICS (SATU KALI SAJA - JANGAN DOUBLE)
metrics_data = {
    "accuracy": acc,
    "precision": prec,
    "recall": rec,
    "f1": f1,
    "loss": test_loss,
    "time": {
        "train_time_sec": train_time,
        "test_time_sec": test_time,
        "avg_inference_ms": avg_inference_per_sample,
        "total_time_sec": total_time
    }
}

with open(f"{OUTPUT_DIR}/metrics.json", "w") as f:
    json.dump(metrics_data, f, indent=4)

# HAPUS BLOK "# ================= SAVE METRICS =================" BERIKUTNYA


# ================= CURVES =================
epochs=[x[0] for x in history]

plt.plot(epochs,[x[1] for x in history],label="train")
plt.plot(epochs,[x[2] for x in history],label="val")
plt.legend(); plt.grid()
plt.savefig(f"{OUTPUT_DIR}/loss_curve.png")
plt.close()

plt.plot(epochs,[x[3] for x in history],label="train acc")
plt.plot(epochs,[x[4] for x in history],label="val acc")
plt.legend(); plt.grid()
plt.savefig(f"{OUTPUT_DIR}/accuracy_curve.png")
plt.close()


# ================= ROC =================
fpr,tpr,_ = roc_curve(test_g,test_prob)
roc_auc = auc(fpr,tpr)

plt.plot(fpr,tpr,label=f"AUC={roc_auc:.4f}")
plt.plot([0,1],[0,1],'--')
plt.legend(); plt.grid()
plt.savefig(f"{OUTPUT_DIR}/roc.png")
plt.close()


# ================= CM =================
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(test_g, test_p)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative","Positive"]
)

plt.figure(figsize=(6,6))
disp.plot(values_format="d")
plt.title("Confusion Matrix")
plt.grid(False)
plt.savefig(f"{OUTPUT_DIR}/confusion_matrix.png")
plt.close()

# ================= SAVE TRAINING LOGS (CSV) =================
log_df = pd.DataFrame(history, columns=["epoch", "train_loss", "val_loss", "train_acc", "val_acc"])
log_df.to_csv(f"{OUTPUT_DIR}/training_log.csv", index=False)

# ================= SAVE DETAILED CLASSIFICATION REPORT =================
from sklearn.metrics import classification_report
report = classification_report(test_g, test_p, target_names=["Negative", "Positive"], output_dict=True)
pd.DataFrame(report).transpose().to_csv(f"{OUTPUT_DIR}/classification_report.csv")

# ================= SAVE PREDICTIONS (Lebih Lengkap untuk Analisis) =================
pred_df = pd.DataFrame({
    "text": test_texts,        # Tambahkan ini agar bisa dibaca manusia
    "actual": test_g,
    "predicted": test_p,
    "probability": test_prob
})
pred_df.to_csv(f"{OUTPUT_DIR}/test_predictions.csv", index=False)

<>:120: SyntaxWarning: invalid escape sequence '\T'
<>:120: SyntaxWarning: invalid escape sequence '\T'
/tmp/ipykernel_3463/1832973146.py:120: SyntaxWarning: invalid escape sequence '\T'
  train_idx = np.load(r"D:\TOPIK RISET\LATIAN\PREPROCESSING\SPLIT_INDEX\train.npy")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LR = 2.88e-06


Epoch 1 | train_loss 0.5780 | val_loss 0.4599 | train_acc 0.6901 | val_acc 0.7824


...

Epoch 25 | train_loss 0.1938 | val_loss 0.3898 | train_acc 0.9244 | val_acc 0.8568


Epoch 26 | train_loss 0.1894 | val_loss 0.3920 | train_acc 0.9269 | val_acc 0.8578


Epoch 27 | train_loss 0.1794 | val_loss 0.4006 | train_acc 0.9308 | val_acc 0.8572
Early stopping triggered at epoch 27



TEST RESULT
0.8628 0.8496530454895914 0.8816 0.8653317628582646


<Figure size 600x600 with 0 Axes>